> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v0, LLM-named 82-name universe). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# Meme / Short-Squeeze Early-Detection Scanner

**Triết lý cốt lõi — tách 2 lớp:**

| Lớp | Đặc tính | Nội dung | Tần suất |
|---|---|---|---|
| **Structural** (không đổi nhanh) | *"ai CÓ THỂ cháy"* | float nhỏ, SI cao, mcap small/mid, fundamentals yếu | 1×/ngày |
| **Ignition** (đổi nhanh) | *"ai ĐANG BẮT ĐẦU cháy"* | social acceleration, borrow-fee spike, RVOL bùng | mỗi 2–5 phút |

> **Market cap KHÔNG phải trigger.** Dùng mcap như *static band* để định nghĩa universe (trần/sàn), **không** dùng làm tín hiệu thời điểm. Nếu lọc "khi mcap đã tăng" là đã trễ. Phát hiện sớm đến từ **trigger layer**, không từ mcap.

Pre-load watchlist các tên *meme-able* TRƯỚC khi có bất kỳ chuyển động giá nào. Khi ignition signal bật → đã sẵn sàng.

**LLM (🤖) chỉ ở Section 4** — biến social text thô → sentiment score có cấu trúc, nằm trong *slow path*. Nó là một feature feeding vào composite score, **không phải decision-maker**. Quyết định (Section 6) là rule-based.

---
## Section 0 — Setup & config

In [ ]:
# Cell 0.0 — CÀI DEPENDENCY VÀO ĐÚNG KERNEL đang chạy notebook
# %pip cài vào sys.executable của kernel hiện tại -> đúng chỗ import, KHÔNG phụ thuộc terminal/env nào.
%pip install -q -U yfinance
%pip install -q pyarrow requests pandas numpy scipy matplotlib
%pip install -q pytrends   # C-proxy: Google Trends (Section 9, experimental)

import sys
print("Kernel Python :", sys.executable)   # xác nhận đúng env (vd .../anaconda3/bin/python)
import importlib
for m in ["yfinance", "pyarrow", "requests", "pandas", "numpy", "scipy"]:
    try:
        mod = importlib.import_module(m)
        print(f"  OK   {m:10s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  FAIL {m:10s} {e}")


In [ ]:
# Cell 0.1 — imports
import os
import re
import json
import time
import logging
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import requests  # ApeWisdom / StockTwits / iBorrowDesk / OpenRouter đều qua HTTP keyless

# Data: yfinance (fundamentals) + Alpaca (universe + RVOL). KHÔNG cần praw / ib_insync nữa.
try:
    import yfinance as yf
except ImportError:
    yf = None

try:
    from alpaca.trading.client import TradingClient
    from alpaca.data.historical import StockHistoricalDataClient
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.timeframe import TimeFrame
except ImportError:
    TradingClient = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("meme_scanner")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# ---- Persistence helpers: ưu tiên parquet, TỰ fallback pickle nếu thiếu pyarrow ----
def save_df(df, path):
    try:
        df.to_parquet(path, index=False)
    except Exception:
        df.to_pickle(Path(str(path) + ".pkl"))

def load_df(path):
    path = Path(path)
    if path.exists():
        try:
            return pd.read_parquet(path)
        except Exception:
            pass
    pk = Path(str(path) + ".pkl")
    if pk.exists():
        return pd.read_pickle(pk)
    return pd.DataFrame()

def cache_exists(path):
    return Path(path).exists() or Path(str(path) + ".pkl").exists()


In [ ]:
# Cell 0.2 — config: tất cả keys / thresholds / weights ở MỘT chỗ để tune
CACHE_DIR = Path("./meme_cache")
CACHE_DIR.mkdir(exist_ok=True)

CONFIG = {
    # ---- API keys (ưu tiên env; fallback = giá trị bạn đưa) ----
    # Chỉ còn cần ALPACA (key+secret) + OpenRouter. KHÔNG cần Reddit, KHÔNG cần IBKR.
    "keys": {
        "alpaca_key":       os.getenv("ALPACA_API_KEY", ""),
        "alpaca_secret":    os.getenv("ALPACA_SECRET_KEY", ""),
        # DeepSeek qua OpenRouter:
        "deepseek_key":     os.getenv("DEEPSEEK_API_KEY", ""),
    },
    # User-Agent dùng cho các HTTP keyless (ApeWisdom / Reddit JSON / iBorrowDesk)
    "http_user_agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    # ---- DeepSeek qua OpenRouter (model slug dạng provider/model) ----
    "deepseek": {
        "model":    os.getenv("DEEPSEEK_MODEL", ""),
        "base_url": os.getenv("DEEPSEEK_BASE_URL", "https://openrouter.ai/api/v1"),
    },
    # ---- Alpaca (paper) ----
    "alpaca": {"paper": True, "endpoint": "https://paper-api.alpaca.markets/v2"},
    # ---- Section 1: sanity LỎNG (không hard-gate short%/float/dtc nữa) ----
    "universe": {
        "min_price":    1.0,
        "mcap_floor":   1e8,    # 100M  -> sàn rộng (giữ KOSS-class)
        "mcap_ceiling": 3e10,   # 30B   -> giữ GME (~12B), loại mega như MSTR (~80B)
    },
    # ---- Ngưỡng THÍCH NGHI (percentile + winner-calibration) thay cho số cố định ----
    "adaptive": {
        "winners_file":     "winners.csv",
        "squeeze_pct":      {"short_pct": 0.60, "dtc": 0.60},          # squeeze leg: ≥ percentile-60 hôm nay
        "momentum_pct":     {"mention_accel": 0.80, "rvol": 0.80},     # momentum leg: ≥ percentile-80 hôm nay
        "winner_floor_pct": 0.10,    # nới ngưỡng để bắt được 10%-thấp-nhất của winners đã gắn nhãn
        "use_winner_calibration": True,
    },
    # ---- Section 3/4: social ----
    "social": {
        "apewisdom_filter":    "all-stocks",  # ApeWisdom đã gộp các sub
        "subreddits":          ["wallstreetbets", "shortsqueeze", "pennystocks"],  # chỉ để lấy TEXT cho LLM
        "reddit_window_hours": 12,
        "mention_roll_days":   25,
        "llm_top_n":    25,         # chỉ chạy LLM trên N tên mention-spike nhất (tiết kiệm cost)
    },
    # ---- Section 5: composite weights ----
    "weights": {
        "short_pct":    1.0,
        "dtc":          0.8,
        "util":         0.8,
        "rvol":         1.2,
        "mention_accel":1.5,
        "borrow_fee_d": 1.3,
        "llm_fomo":     1.0,
        "otm_call_vol": 0.6,
        "float_pen":    0.7,   # trừ
        "mcap_pen":     0.5,   # trừ
    },
    # ---- Section 6: trigger = xác nhận intraday (chọn lọc cơ học đã do leg thích nghi lo) ----
    "trigger": {
        "min_rvol":           3.0,    # "có thật sự đang chạy hôm nay" - khá phổ quát
        "require_above_vwap": True,
        "require_borrow_up":  False,
    },
    "webhook_url": os.getenv("ALERT_WEBHOOK_URL", ""),
}

# Header browser-like dùng chung cho mọi HTTP keyless (ApeWisdom / Reddit JSON / iBorrowDesk)
CONFIG["http_headers"] = {
    "User-Agent":      CONFIG["http_user_agent"],
    "Accept":          "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
}

CONFIG

In [ ]:
# Cell 0.3 — SMOKE TEST: verify schema từng nguồn keyless trước khi chạy full pipeline.
# Chạy cell này TRƯỚC. Nếu thấy ✅ hết là pipeline chạy được; chỗ nào ❌ thì sửa parser/filter tương ứng.
def _smoke():
    ua = CONFIG["http_headers"]

    # --- 1) iBorrowDesk: kỳ vọng có 'real_time'/'daily', mỗi entry có 'fee' + 'available' ---
    print("=" * 60, "\n[1] iBorrowDesk (borrow fee + shortable) — GME")
    try:
        r = requests.get("https://iborrowdesk.com/api/ticker/GME", headers=ua, timeout=10)
        print("   HTTP", r.status_code)
        js = r.json()
        print("   top-level keys:", list(js.keys()))
        series = js.get("real_time") or js.get("daily") or []
        if series:
            latest = series[-1]
            print("   entry keys:", list(latest.keys()))
            print(f"   ✅ fee={latest.get('fee')}  available={latest.get('available')}  time={latest.get('time')}")
        else:
            print("   ❌ không có real_time/daily — kiểm tra lại schema")
    except Exception as e:
        print("   ❌ lỗi:", e)

    # --- 2) ApeWisdom: kỳ vọng 'results' = list, mỗi item có 'ticker','mentions','mentions_24h_ago' ---
    print("=" * 60, "\n[2] ApeWisdom (mention count) — filter =", CONFIG["social"]["apewisdom_filter"])
    try:
        url = f"https://apewisdom.io/api/v1.0/filter/{CONFIG['social']['apewisdom_filter']}/page/1"
        r = requests.get(url, headers=ua, timeout=10)
        print("   HTTP", r.status_code)
        js = r.json()
        results = js.get("results", [])
        print(f"   pages={js.get('pages')}  results={len(results)}")
        if results:
            it = results[0]
            print("   item keys:", list(it.keys()))
            print(f"   ✅ top: {it.get('ticker')} mentions={it.get('mentions')} 24h_ago={it.get('mentions_24h_ago')}")
        else:
            print("   ❌ results rỗng — thử đổi filter sang 'wallstreetbets'")
    except Exception as e:
        print("   ❌ lỗi:", e)

    # --- 3) Reddit public JSON (best-effort; hiện hay 403 -> KHÔNG sao, đã có StockTwits) ---
    print("=" * 60, "\n[3] Reddit public JSON — r/wallstreetbets (best-effort)")
    try:
        r = requests.get("https://www.reddit.com/r/wallstreetbets/new.json?limit=5", headers=ua, timeout=10)
        print("   HTTP", r.status_code)
        children = r.json().get("data", {}).get("children", []) if r.status_code == 200 else []
        if children:
            print(f"   ✅ {len(children)} post; ví dụ:", children[0]["data"].get("title", "")[:60])
        else:
            print("   ⚠️ Reddit chặn (403) — bỏ qua, text cho LLM lấy từ StockTwits ở [3b].")
    except Exception as e:
        print("   ⚠️ Reddit lỗi (bỏ qua):", e)

    # --- 3b) StockTwits: NGUỒN TEXT CHÍNH cho LLM ---
    print("=" * 60, "\n[3b] StockTwits — GME (text cho LLM)")
    try:
        r = requests.get("https://api.stocktwits.com/api/2/streams/symbol/GME.json", headers=ua, timeout=10)
        print("   HTTP", r.status_code, "(403/429 = StockTwits chặn -> LLM tạm không có text)")
        msgs = r.json().get("messages", []) if r.status_code == 200 else []
        if msgs:
            print(f"   ✅ {len(msgs)} message; ví dụ:", (msgs[0].get("body") or "")[:70])
        else:
            print("   ❌ không lấy được message")
    except Exception as e:
        print("   ❌ lỗi:", e)

    # --- 4) OpenRouter: model slug có tồn tại + key hợp lệ không ---
    print("=" * 60, "\n[4] OpenRouter — model:", CONFIG["deepseek"]["model"], "(gọi trực tiếp, không cần Cell 4.2)")
    try:
        _u = CONFIG["deepseek"]["base_url"].rstrip("/") + "/chat/completions"
        _pl = {"model": CONFIG["deepseek"]["model"],
               "messages": [{"role": "user", "content": "Reply with the single word: OK"}],
               "max_tokens": 5, "temperature": 0}
        _h = {"Authorization": f"Bearer {CONFIG['keys']['deepseek_key']}", "Content-Type": "application/json"}
        _r = requests.post(_u, json=_pl, headers=_h, timeout=30)
        print("   HTTP", _r.status_code, "(200=OK; 404=sai tên model; 401=sai key)")
        if _r.status_code == 200:
            print("   ✅", _r.json()["choices"][0]["message"]["content"][:60])
        else:
            print("   ❌", _r.text[:160])
    except Exception as e:
        print("   ❌ lỗi:", e)

    # --- 5) Alpaca: key+secret auth được không ---
    print("=" * 60, "\n[5] Alpaca (auth)")
    try:
        import requests as _rq
        h = {"APCA-API-KEY-ID": CONFIG["keys"]["alpaca_key"],
             "APCA-API-SECRET-KEY": CONFIG["keys"]["alpaca_secret"]}
        r = _rq.get("https://paper-api.alpaca.markets/v2/account", headers=h, timeout=10)
        print("   HTTP", r.status_code, "(200 = OK; 401/403 = sai key/secret)")
        if r.status_code == 200:
            acc = r.json()
            print(f"   ✅ account status={acc.get('status')}  buying_power={acc.get('buying_power')}")
    except Exception as e:
        print("   ❌ lỗi:", e)
    print("=" * 60)

_smoke()

---
### Cell 0.4 — Ngưỡng THÍCH NGHI + Winners feedback loop

Thay vì số cố định (`dtc>2`, `short%>15%`), ngưỡng được tính lại mỗi lần chạy:
- **percentile của phân phối hôm nay** → tự co giãn theo regime thị trường.
- **winner-calibration**: tên nào ĐÃ chạy/squeeze bạn gắn `add_winner("WEN")`; ngưỡng tự **nới xuống đủ thấp để từng bắt được nó** (lấy percentile-10 của winners). Đây là feedback loop bạn nói.

In [ ]:
# Cell 0.4 — Adaptive thresholds + winners feedback loop
WINNERS_FILE  = CACHE_DIR / CONFIG["adaptive"]["winners_file"]
HEAT_HISTORY  = CACHE_DIR / "heat_history.parquet"   # nguồn point-in-time để hiệu chỉnh winners

def load_winners():
    if WINNERS_FILE.exists():
        return pd.read_csv(WINNERS_FILE)
    return pd.DataFrame(columns=["ticker", "run_date", "note"])

def add_winner(ticker, run_date=None, note=""):
    """Gắn nhãn 1 tên ĐÃ chạy/squeeze -> feed vào hiệu chỉnh ngưỡng. Gọi tay khi thấy 1 tên nổ."""
    w = load_winners()
    run_date = run_date or datetime.now(timezone.utc).strftime("%Y-%m-%d")
    w = pd.concat([w, pd.DataFrame([{"ticker": ticker.upper(), "run_date": run_date, "note": note}])],
                  ignore_index=True).drop_duplicates(subset=["ticker", "run_date"])
    w.to_csv(WINNERS_FILE, index=False)
    log.info("Winner +%s (%s). Tổng winners: %d", ticker.upper(), run_date, len(w))
    return w

def _pct(series, q):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    return float(np.nanpercentile(s, q * 100)) if len(s) else np.nan

def winner_feature_values(col, wset, cur_df):
    """Giá trị feature của winners: ưu tiên point-in-time từ heat_history, fallback giá trị hiện tại."""
    if not wset:
        return []
    if HEAT_HISTORY.exists() or Path(str(HEAT_HISTORY) + ".pkl").exists():
        hist = load_df(HEAT_HISTORY)
        if len(hist) and col in hist:
            v = pd.to_numeric(hist.loc[hist["ticker"].isin(wset), col], errors="coerce").dropna().tolist()
            if len(v) >= 3:
                return v
    if col in cur_df:
        return pd.to_numeric(cur_df.loc[cur_df["ticker"].isin(wset), col], errors="coerce").dropna().tolist()
    return []

def adaptive_threshold(universe_series, q_universe, winner_values=None):
    """Ngưỡng = percentile cross-section HÔM NAY, nhưng KHÔNG cao hơn mức từng bắt được winners."""
    thr = _pct(universe_series, q_universe)
    if winner_values is not None and len(winner_values) >= 3 and CONFIG["adaptive"]["use_winner_calibration"]:
        wfloor = _pct(winner_values, CONFIG["adaptive"]["winner_floor_pct"])
        if not np.isnan(wfloor):
            thr = wfloor if np.isnan(thr) else min(thr, wfloor)
    return thr

print("Adaptive helpers sẵn sàng. Winners hiện có:", len(load_winners()))


---
## Section 1 — Universe builder (HƯỚNG B: seed + trending, bounded)

Thay vì quét vài nghìn mã (giết yfinance), ta dựng universe **có chủ đích, gọn (~100 tên)**:
1. **Seed tĩnh** các tên small/mid-cap float thấp / SI cao hay được nhắc.
2. **ApeWisdom trending** (WSB + shortsqueeze) — tên đang được chú ý.
3. **Validate** còn tradable trên Alpaca (REST, không cần alpaca-py) — loại tên đã huỷ niêm yết.

Rồi mới lấy fundamentals trên tập gọn này → yfinance chịu nổi. Lọc structural ở Cell 1.3.

In [ ]:
# Cell 1.1 — Universe HƯỚNG B: seed tĩnh + ApeWisdom trending, validate qua Alpaca REST
ALPACA_DATA_HEADERS = {
    "APCA-API-KEY-ID":     CONFIG["keys"]["alpaca_key"],
    "APCA-API-SECRET-KEY": CONFIG["keys"]["alpaca_secret"],
}
APEWISDOM_TREND_URL = "https://apewisdom.io/api/v1.0/filter/{filter}/page/{page}"

# Seed tĩnh: small/mid-cap float thấp / SI cao hay được nhắc. CẬP NHẬT TAY định kỳ.
SEED_SQUEEZE = [
    # nhóm 1 - cựu binh sentiment/high-beta
    "GME","AMC","BB","KOSS",
    # nhóm 2 - bán lẻ & tiêu dùng / short-squeeze
    "WEN","BYND","TUP","CVNA","UPST",
    # nhóm 3 - crypto-proxy
    "MSTR","MARA","RIOT","BTBT","SDIG","HOLO",
    # nhóm 4 - AI speculation
    "BE","SOUN","SMCI","RGTI","QUBT","IONQ","LAES",
    # các tên float thấp / squeeze khác
    "CLOV","ATER","MULN","GNS","DJT","SAVA","FFIE","NKLA","RILY","LUNR",
    "ASTS","ACHR","RDDT","WOLF","SERV","KSCP","REKR","CARM","PRPL","BIGC",
    "FUBO","GETY","ROOT","OPEN","WKHS","PHUN","CELH","HOOD","SOFI","PLTR",
    # mở rộng 2025-2026: EV/H2, nuclear/AI-power, crypto-miner, high-short
    "LCID","RIVN","CHPT","PLUG","FCEL","BLNK","RUN","NIO","XPEV","LI",
    "SPCE","RKLB","OKLO","SMR","NNE","VST","CIFR","WULF","BITF","HUT",
    "CLSK","APLD","CORZ","IREN","GRAB","DNA","JOBY","EVGO","QS","CHPT",
    "ENVX","AUR","LAZR","RKT","UPWK","DKNG","RDFN","BMBL","ALLY","Z",
]

def alpaca_tradable_map():
    """1 call REST -> dict symbol -> asset (tradable US equities). Không cần alpaca-py."""
    try:
        r = requests.get(CONFIG["alpaca"]["endpoint"] + "/assets",
                         params={"status": "active", "asset_class": "us_equity"},
                         headers=ALPACA_DATA_HEADERS, timeout=30)
        r.raise_for_status()
        return {a["symbol"]: a for a in r.json()
                if a.get("tradable") and a.get("exchange") in ("NYSE", "NASDAQ", "ARCA", "AMEX", "BATS")}
    except Exception as e:
        log.warning("Alpaca /assets lỗi (%s) -> bỏ bước validate.", e)
        return {}

def apewisdom_trending(filters=("wallstreetbets", "shortsqueeze"), pages=2):
    """Tên đang được nhắc nhiều -> ứng viên ignition (lọc structural sau ở Cell 1.3)."""
    names = []
    for flt in filters:
        for page in range(1, pages + 1):
            try:
                r = requests.get(APEWISDOM_TREND_URL.format(filter=flt, page=page),
                                 headers=CONFIG["http_headers"], timeout=8)
                if r.status_code != 200:
                    break
                for it in r.json().get("results", []):
                    tk = (it.get("ticker") or "").upper()
                    if tk:
                        names.append(tk)
                time.sleep(0.3)
            except Exception as e:
                log.debug("apewisdom trend %s p%d: %s", flt, page, e)
    return names

def get_candidate_universe(max_names=120):
    # ưu tiên seed trước, rồi trending; dedupe giữ thứ tự
    ordered = list(dict.fromkeys(SEED_SQUEEZE + apewisdom_trending()))
    ordered = [c for c in ordered if c.isalpha() and 1 <= len(c) <= 5]
    tradable = alpaca_tradable_map()
    if tradable:
        before = len(ordered)
        ordered = [c for c in ordered if c in tradable]
        log.info("Validate Alpaca tradable: giữ %d/%d", len(ordered), before)
    out = ordered[:max_names]
    log.info("Candidate universe (huong B): %d ten", len(out))
    return out

tickers = get_candidate_universe()
print(tickers)


In [ ]:
# Cell 1.2 — fundamentals: FINVIZ (chính, đáng tin cho float/short%) + yfinance (fallback), cache
FUND_CACHE = CACHE_DIR / f"fundamentals_{datetime.now():%Y%m%d}.parquet"
FINVIZ_URL = "https://finviz.com/quote.ashx?t={sym}&p=d"

# yfinance fallback: session UA-browser + backoff 429
_YF_SESSION = requests.Session()
_YF_SESSION.headers.update(CONFIG["http_headers"])

def _yf_info(sym, retries=3):
    for k in range(retries):
        try:
            try:
                info = yf.Ticker(sym, session=_YF_SESSION).info
            except TypeError:
                info = yf.Ticker(sym).info
            return info or {}
        except Exception as e:
            if any(x in str(e) for x in ("429", "Too Many", "crumb")):
                time.sleep(min(2 ** k, 12)); continue
            return {}
    return {}

def _num(s):
    """Parse số kiểu Finviz: '1.2B','850.00M','15.30%','-' -> float (hoặc NaN)."""
    if s is None: return np.nan
    s = str(s).replace(",", "").replace("%", "").strip()
    if s in ("-", "", "N/A"): return np.nan
    mult = 1.0
    if s and s[-1] in "BMK":
        mult = {"B": 1e9, "M": 1e6, "K": 1e3}[s[-1]]; s = s[:-1]
    try: return float(s) * mult
    except Exception: return np.nan

def _finviz_fundamentals(sym):
    r = requests.get(FINVIZ_URL.format(sym=sym), headers=CONFIG["http_headers"], timeout=10)
    if r.status_code != 200:
        return None
    # bảng snapshot: các ô class="snapshot-td2*" xen kẽ nhãn / giá trị
    cells = re.findall(r'class="snapshot-td2[^"]*"[^>]*>(.*?)</td>', r.text, re.S)
    cells = [re.sub(r"<.*?>", "", c).strip() for c in cells]
    if len(cells) < 6:
        return None
    d = {cells[i]: cells[i + 1] for i in range(0, len(cells) - 1, 2)}
    # 'Short Float / Ratio' có thể gộp 1 ô: '24.15% / 3.21'
    sf = d.get("Short Float / Ratio") or d.get("Short Float") or ""
    spct = dtc = np.nan
    if "/" in sf:
        a, b = sf.split("/", 1); spct = _num(a); dtc = _num(b)
    else:
        spct = _num(sf)
    if np.isnan(dtc):
        dtc = _num(d.get("Short Ratio"))
    return {
        "ticker":          sym,
        "price":           _num(d.get("Price")),
        "market_cap":      _num(d.get("Market Cap")),
        "float_shares":    _num(d.get("Shs Float")),
        "short_pct_float": (spct / 100.0) if not np.isnan(spct) else np.nan,
        "shares_short":    np.nan,
        "days_to_cover":   dtc,
        "sector":          None,
        "profit_margin":   (_num(d.get("Profit Margin")) / 100.0) if d.get("Profit Margin") else np.nan,
    }

def _yf_fundamentals(sym):
    info = _yf_info(sym)
    if not info: return None
    return {
        "ticker": sym,
        "price": info.get("currentPrice") or info.get("previousClose"),
        "market_cap": info.get("marketCap"),
        "float_shares": info.get("floatShares"),
        "short_pct_float": info.get("shortPercentOfFloat"),
        "shares_short": info.get("sharesShort"),
        "days_to_cover": info.get("shortRatio"),
        "sector": info.get("sector"),
        "profit_margin": info.get("profitMargins"),
    }

def fetch_fundamentals(symbols, force=False):
    if cache_exists(FUND_CACHE) and not force:
        log.info("Load fundamentals từ cache"); return load_df(FUND_CACHE)
    rows, src_finviz, src_yf = [], 0, 0
    for i, sym in enumerate(symbols):
        rec = None
        try:
            rec = _finviz_fundamentals(sym)
        except Exception as e:
            log.debug("finviz %s: %s", sym, e)
        if rec and not np.isnan(rec["market_cap"]):
            src_finviz += 1
        elif yf is not None:                       # fallback yfinance
            rec = _yf_fundamentals(sym)
            if rec: src_yf += 1
        if rec:
            rows.append(rec)
        time.sleep(0.7)
        if i % 25 == 0:
            log.info("fundamentals %d/%d", i, len(symbols))
    df = pd.DataFrame(rows)
    if len(df):
        save_df(df, FUND_CACHE)
        log.info("Cached %d rows (finviz=%d, yfinance=%d)", len(df), src_finviz, src_yf)
    else:
        log.warning("fundamentals RỖNG — cả Finviz lẫn yfinance fail. "
                    "Có thể bị chặn IP; thử lại sau, hoặc nâng yfinance + restart kernel.")
    return df

fund = fetch_fundamentals(tickers)
fund


In [ ]:
# Cell 1.3 — sanity LỎNG -> candidate pool (leg phân loại sau ở Section 5, bằng ngưỡng thích nghi)
u = CONFIG["universe"]
_BASE = ["ticker", "price", "market_cap", "float_shares", "short_pct_float", "days_to_cover"]

def build_meme_able(df):
    if df is None or len(df) == 0 or "ticker" not in getattr(df, "columns", []):
        log.warning("fundamentals rỗng -> candidate pool RỖNG. Kiểm tra Cell 1.2.")
        return pd.DataFrame(columns=_BASE)
    d = df.copy()
    if "price" in d and "market_cap" in d:
        d = d.dropna(subset=["price", "market_cap"])
        d = d[(d["price"] > u["min_price"]) &
              (d["market_cap"] > u["mcap_floor"]) &
              (d["market_cap"] < u["mcap_ceiling"])]
    log.info("Candidate pool (sanity lỏng): %d tên — leg sẽ chọn ở Section 5", len(d))
    return d.reset_index(drop=True)

meme_able = build_meme_able(fund)
if len(meme_able) == 0:
    log.warning("pool rỗng -> tạm dùng %d candidate tickers để demo.", len(tickers))
    watchlist = list(tickers)
    meme_able = pd.DataFrame({"ticker": watchlist})
else:
    watchlist = meme_able["ticker"].tolist()
print("candidate pool size:", len(watchlist))
meme_able.head(30) if len(meme_able) else meme_able


---
## Section 2 — Borrow & liquidity layer (iBorrowDesk keyless + Alpaca RVOL)

**Δ (thay đổi) mới là tín hiệu, không phải level.** Borrow-fee *spike*, shortable *tụt* → ignition.

> iBorrowDesk phục vụ lại đúng data short-availability của IBKR nhưng **không cần Gateway / login / API key**.

In [ ]:
# Cell 2.1 — borrow fee + shortable shares KEYLESS qua iBorrowDesk
IBORROWDESK_URL = "https://iborrowdesk.com/api/ticker/{sym}"

def fetch_borrow(symbols, pause=0.25):
    """Pull borrow fee (%/năm) + shortable shares từ iBorrowDesk. Không cần IBKR Gateway."""
    rows = []
    for sym in symbols:
        try:
            r = requests.get(IBORROWDESK_URL.format(sym=sym), headers=CONFIG["http_headers"], timeout=8)
            if r.status_code != 200:
                continue
            js = r.json()
            fee   = js.get("latest_fee")
            avail = js.get("latest_available")
            if fee is None or avail is None:        # fallback: entry mới nhất trong series
                series = js.get("real_time") or js.get("daily") or []
                if series:
                    fee   = series[-1].get("fee", fee)
                    avail = series[-1].get("available", avail)
            rows.append({"ticker": sym, "borrow_fee": fee, "shortable_shares": avail,
                         "ts": datetime.now(timezone.utc)})
            time.sleep(pause)
        except Exception as e:
            log.debug("borrow skip %s: %s", sym, e)
    df = pd.DataFrame(rows)
    log.info("Borrow (iBorrowDesk): %d/%d mã có data", len(df), len(symbols))
    return df

borrow = fetch_borrow(watchlist)
borrow


In [ ]:
# Cell 2.2 — RVOL = volume hôm nay / avg lookback (Alpaca REST data API, feed IEX free)
def fetch_rvol(symbols, lookback=20):
    base = "https://data.alpaca.markets/v2/stocks/{sym}/bars"
    start = (datetime.now(timezone.utc) - timedelta(days=lookback * 3)).strftime("%Y-%m-%d")
    rows = []
    for sym in symbols:
        try:
            r = requests.get(base.format(sym=sym),
                params={"timeframe": "1Day", "start": start, "limit": lookback + 5,
                        "feed": "iex", "adjustment": "raw"},
                headers=ALPACA_DATA_HEADERS, timeout=10)
            if r.status_code != 200:
                log.debug("rvol %s HTTP %d", sym, r.status_code)
                continue
            bars = r.json().get("bars") or []
            if len(bars) > lookback:
                vols = np.array([b["v"] for b in bars], dtype=float)
                rvol = vols[-1] / np.mean(vols[-lookback - 1:-1])
                rows.append({"ticker": sym, "rvol": float(rvol),
                             "price": bars[-1]["c"], "vwap": bars[-1].get("vw", bars[-1]["c"])})
            time.sleep(0.1)
        except Exception as e:
            log.debug("rvol skip %s: %s", sym, e)
    df = pd.DataFrame(rows)
    log.info("RVOL (Alpaca): %d/%d ma co bars", len(df), len(symbols))
    return df

rvol_df = fetch_rvol(watchlist)
rvol_df


In [ ]:
# Cell 2.3 — snapshot có timestamp -> tính Δ (borrow fee spike, shortable tụt)
SNAP_FILE = CACHE_DIR / "borrow_snapshots.parquet"

def append_snapshot(df_now):
    df_now = df_now.copy()
    df_now["ts"] = datetime.now(timezone.utc)
    hist = load_df(SNAP_FILE) if cache_exists(SNAP_FILE) else pd.DataFrame()
    df_all = pd.concat([hist, df_now], ignore_index=True) if len(hist) else df_now
    save_df(df_all, SNAP_FILE)
    return df_all

def compute_deltas(df_all):
    """Δ borrow_fee và Δ shortable so với snapshot trước đó của cùng ticker."""
    df_all = df_all.sort_values(["ticker", "ts"])
    df_all["borrow_fee_d"] = df_all.groupby("ticker")["borrow_fee"].diff()
    df_all["shortable_d"]  = df_all.groupby("ticker")["shortable_shares"].diff()
    return df_all.groupby("ticker").tail(1)

# Lần ĐẦU borrow_fee_d = NaN (chưa có snapshot trước) -> bình thường, có Δ từ vòng 2.
if len(borrow):
    deltas = compute_deltas(append_snapshot(borrow))
else:
    deltas = pd.DataFrame(columns=["ticker", "borrow_fee", "shortable_shares", "borrow_fee_d", "shortable_d"])
deltas


---
## Section 3 — Social ingestion (raw, KEYLESS)

**Acceleration quan trọng hơn level.** Đạo hàm của mentions > số tuyệt đối.

> Đếm mention lấy từ **ApeWisdom** (đã gộp sẵn các sub, không key). Text thô cho LLM lấy từ **Reddit public JSON** (`/new.json`, chỉ cần User-Agent, không key). Không dùng PRAW.

In [ ]:
# Cell 3.1 — Social mentions KEYLESS: ApeWisdom (đếm) + StockTwits/Reddit (text cho LLM)
CASHTAG_RE = re.compile(r"\$([A-Z]{1,5})\b")
TICKER_RE  = re.compile(r"\b([A-Z]{2,5})\b")

APEWISDOM_URL   = "https://apewisdom.io/api/v1.0/filter/{filter}/page/{page}"
REDDIT_JSON_URL = "https://www.reddit.com/r/{sub}/new.json?limit=100"
STOCKTWITS_URL  = "https://api.stocktwits.com/api/2/streams/symbol/{sym}.json"

def apewisdom_mentions(watch_set, max_pages=3):
    """ApeWisdom đã scrape + gộp Reddit sẵn -> mentions + mentions_24h_ago (tính Δ ngay)."""
    flt = CONFIG["social"]["apewisdom_filter"]
    rows = []
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(APEWISDOM_URL.format(filter=flt, page=page),
                             headers=CONFIG["http_headers"], timeout=8)
            if r.status_code != 200:
                break
            for it in r.json().get("results", []):
                tk = (it.get("ticker") or "").upper()
                if tk in watch_set:
                    rows.append({"ticker": tk,
                                 "mentions": float(it.get("mentions", 0) or 0),
                                 "mentions_prev": float(it.get("mentions_24h_ago", 0) or 0)})
            time.sleep(0.3)
        except Exception as e:
            log.debug("apewisdom page %d: %s", page, e)
    return pd.DataFrame(rows)

def reddit_json_posts(watch_set):
    """Best-effort: Reddit public JSON (hiện hay bị 403). Trả dict ticker -> [posts]."""
    cutoff = time.time() - CONFIG["social"]["reddit_window_hours"] * 3600
    posts = {}
    for sub in CONFIG["social"]["subreddits"]:
        try:
            r = requests.get(REDDIT_JSON_URL.format(sub=sub), headers=CONFIG["http_headers"], timeout=10)
            if r.status_code == 403:
                r = requests.get(REDDIT_JSON_URL.format(sub=sub).replace("www.reddit", "old.reddit"),
                                 headers=CONFIG["http_headers"], timeout=10)
            if r.status_code != 200:
                log.debug("reddit json %s -> HTTP %d (bỏ qua, dùng StockTwits)", sub, r.status_code)
                continue
            for child in r.json().get("data", {}).get("children", []):
                d = child.get("data", {})
                if d.get("created_utc", 0) < cutoff:
                    continue
                text = f"{d.get('title','')} {d.get('selftext','')}".strip()
                found = set(CASHTAG_RE.findall(text.upper())) | (set(TICKER_RE.findall(text.upper())) & watch_set)
                for tk in found:
                    if tk in watch_set:
                        posts.setdefault(tk, []).append(text[:600])
            time.sleep(1.0)
        except Exception as e:
            log.debug("reddit json %s: %s", sub, e)
    return posts

def stocktwits_posts(symbols, max_msgs=15):
    """Text thô từ StockTwits (keyless) -> dict ticker -> [bodies]. Nguồn text CHÍNH cho LLM."""
    out = {}
    for sym in symbols:
        try:
            r = requests.get(STOCKTWITS_URL.format(sym=sym), headers=CONFIG["http_headers"], timeout=8)
            if r.status_code != 200:
                continue
            bodies = [m.get("body", "") for m in r.json().get("messages", [])[:max_msgs] if m.get("body")]
            if bodies:
                out[sym] = bodies
            time.sleep(0.4)
        except Exception as e:
            log.debug("stocktwits posts %s: %s", sym, e)
    return out

def fetch_social_mentions(watch_set):
    """Count từ ApeWisdom + text từ StockTwits (chính) / Reddit (best-effort)."""
    ape   = apewisdom_mentions(watch_set)
    posts = reddit_json_posts(watch_set)           # best-effort, có thể rỗng
    active = (ape.sort_values("mentions", ascending=False)["ticker"].head(30).tolist()
              if len(ape) else list(watch_set)[:30])
    for tk, msgs in stocktwits_posts(active).items():   # StockTwits bù text cho LLM
        posts.setdefault(tk, []).extend(msgs)
    if len(ape):
        ape["posts"] = ape["ticker"].map(lambda t: posts.get(t, []))
        df = ape
    else:
        df = pd.DataFrame([{"ticker": t, "mentions": float(len(p)), "mentions_prev": np.nan, "posts": p}
                           for t, p in posts.items()])
    n_text = sum(1 for p in posts.values() if p)
    log.info("Social: %d tickers (ApeWisdom=%d, có-text=%d)", len(df), len(ape), n_text)
    return df

reddit_df = fetch_social_mentions(set(watchlist))
reddit_df


In [ ]:
# Cell 3.2 — StockTwits: message volume + watchlist count (bổ sung cho mention count)
def stocktwits_metrics(symbols):
    rows = []
    for sym in symbols:
        try:
            r = requests.get(f"https://api.stocktwits.com/api/2/streams/symbol/{sym}.json",
                             headers=CONFIG["http_headers"], timeout=8)
            if r.status_code != 200:
                continue
            js = r.json()
            msgs = js.get("messages", [])
            bull = sum(1 for m in msgs if (m.get("entities", {}).get("sentiment") or {}).get("basic") == "Bullish")
            rows.append({
                "ticker": sym,
                "st_msg_volume": len(msgs),
                "st_watchlist":  js.get("symbol", {}).get("watchlist_count", np.nan),
                "st_bull_ratio": bull / max(len(msgs), 1),
            })
            time.sleep(0.4)
        except Exception as e:
            log.debug("stocktwits skip %s: %s", sym, e)
    return pd.DataFrame(rows)

st_df = stocktwits_metrics(watchlist[:50])  # rate-limited, giới hạn lượt
st_df


In [ ]:
# Cell 3.3 — mention_z (level) + mention_accel (Δmention, z-score cross-sectional)
MENTION_HIST = CACHE_DIR / "mention_history.parquet"

def update_mention_features(social_df, st_df=None):
    now = datetime.now(timezone.utc)
    cur = pd.DataFrame({"ticker": watchlist}).set_index("ticker")
    cur["mentions"] = 0.0
    cur["mentions_prev"] = np.nan
    if social_df is not None and len(social_df):
        s = social_df.set_index("ticker")
        idx = s.index.intersection(cur.index)
        cur.loc[idx, "mentions"] = s.loc[idx, "mentions"]
        if "mentions_prev" in s:
            cur.loc[idx, "mentions_prev"] = s.loc[idx, "mentions_prev"]
    if st_df is not None and len(st_df):
        cur["mentions"] = cur["mentions"].add(
            st_df.set_index("ticker")["st_msg_volume"].reindex(cur.index).fillna(0), fill_value=0)
    cur = cur.reset_index()
    cur["ts"] = now

    # persist level history (để rolling z dài hạn về sau)
    hist = load_df(MENTION_HIST) if cache_exists(MENTION_HIST) else pd.DataFrame()
    hist = pd.concat([hist, cur[["ticker", "mentions", "ts"]]], ignore_index=True)
    save_df(hist, MENTION_HIST)

    roll_days = CONFIG["social"]["mention_roll_days"]
    hg = hist.sort_values("ts").groupby("ticker")
    feats = []
    for _, row in cur.iterrows():
        tk = row["ticker"]
        series = np.log1p(hg.get_group(tk)["mentions"].astype(float).values) if tk in hg.groups \
                 else np.array([np.log1p(row["mentions"])])
        # level z
        if len(series) >= 2:
            window = series[-roll_days:]
            mu, sd = window.mean(), window.std(ddof=0)
            z = (series[-1] - mu) / sd if sd > 0 else 0.0
        else:
            z = 0.0
        # accel: ưu tiên Δ24h của ApeWisdom (chạy ngay từ lần đầu), fallback đạo hàm bậc 2
        if not np.isnan(row["mentions_prev"]):
            accel_raw = np.log1p(row["mentions"]) - np.log1p(row["mentions_prev"])
        elif len(series) >= 3:
            accel_raw = series[-1] - 2 * series[-2] + series[-3]
        else:
            accel_raw = 0.0
        feats.append({"ticker": tk, "mention_z": z, "accel_raw": accel_raw,
                      "mentions_now": float(row["mentions"])})

    out = pd.DataFrame(feats)
    # chuẩn hóa accel cross-sectional -> trigger so với 1.5 = 1.5 sigma so với các tên khác
    sd = out["accel_raw"].std(ddof=0)
    out["mention_accel"] = (out["accel_raw"] - out["accel_raw"].mean()) / sd if sd and sd > 0 else 0.0
    return out

mention_feat = update_mention_features(reddit_df)
mention_feat.sort_values("mention_accel", ascending=False)

---
## Section 4 — 🤖 LLM sentiment enrichment (DeepSeek, slow path)

**LLM vào ĐÚNG ở đây:** social text thô → sentiment score có cấu trúc. Chạy mỗi vài phút, **không** mỗi tick. Chỉ chạy trên top-N tên mention-spike để tiết kiệm cost. Đây là *feature*, không phải decision-maker.

In [ ]:
# Cell 4.1 — chọn top-N tên mention spike + gom post thô
def select_llm_candidates(mention_feat, reddit_df, top_n):
    top = mention_feat.sort_values("mention_accel", ascending=False).head(top_n)
    posts_map = reddit_df.set_index("ticker")["posts"].to_dict() if len(reddit_df) else {}
    batch = []
    for tk in top["ticker"]:
        raw = posts_map.get(tk, [])
        if raw:
            batch.append({"ticker": tk, "text": "\n---\n".join(raw[:8])})
    log.info("LLM candidates có text: %d", len(batch))
    return batch

llm_batch = select_llm_candidates(mention_feat, reddit_df, CONFIG["social"]["llm_top_n"])
len(llm_batch)

In [ ]:
# Cell 4.2 — 🤖 DeepSeek: PHÂN TÍCH CONTEXT (bull/bear + concept + risk), KHÔNG phán đoán meme
LLM_SYSTEM = (
    "You are a financial market-context analyst. You read raw social posts (Reddit/StockTwits) about ONE ticker. "
    "Your job is NOT to decide if it is a meme stock and NOT to give trading advice — that logic belongs to a "
    "separate quant pipeline. ONLY characterize, strictly from the crowd's words:\n"
    "1) prevailing sentiment regime (bull/bear/neutral) + confidence;\n"
    "2) the narrative/concepts driving chatter (e.g. AI datacenter power, crypto beta, short squeeze, earnings, dilution);\n"
    "3) concrete bull_points and bear_points actually mentioned;\n"
    "4) explicit risk_flags (dilution, debt, liquidity, bankruptcy, SEC, offering, reverse split);\n"
    "5) sentiment_intensity 0-1 (how heated/coordinated the chatter is);\n"
    "6) a neutral 1-2 sentence summary of the context.\n"
    "Return STRICT JSON only, no prose. Schema:\n"
    "{\"market_regime\": \"bull\"|\"bear\"|\"neutral\", \"regime_confidence\": 0-1, "
    "\"narrative\": str, \"key_concepts\": [str], \"bull_points\": [str], \"bear_points\": [str], "
    "\"risk_flags\": [str], \"sentiment_intensity\": 0-1, \"summary\": str}"
)

def call_deepseek(ticker, text):
    if not CONFIG["keys"]["deepseek_key"]:
        return {"market_regime": "neutral", "regime_confidence": 0.0, "narrative": "", "key_concepts": [],
                "bull_points": [], "bear_points": [], "risk_flags": [], "sentiment_intensity": 0.0, "summary": ""}
    url = CONFIG["deepseek"]["base_url"].rstrip("/") + "/chat/completions"
    payload = {
        "model": CONFIG["deepseek"]["model"],
        "messages": [{"role": "system", "content": LLM_SYSTEM},
                     {"role": "user", "content": f"TICKER: {ticker}\nPOSTS:\n{text}"}],
        "temperature": 0.2,
        "response_format": {"type": "json_object"},
        "max_tokens": 600,
    }
    headers = {"Authorization": f"Bearer {CONFIG['keys']['deepseek_key']}", "Content-Type": "application/json"}
    r = requests.post(url, json=payload, headers=headers, timeout=40)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


In [ ]:
# Cell 4.3 — parse an toàn -> feature + llm_score + IN summary context (bull/bear)
def safe_parse_json(raw):
    if isinstance(raw, dict): return raw
    if not isinstance(raw, str): return None
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, re.DOTALL)
        if m:
            try: return json.loads(m.group(0))
            except Exception: return None
    return None

_REGIME = {"bull": 1.0, "bear": -1.0, "neutral": 0.0}

def run_llm_enrichment(batch):
    rows = []
    for item in batch:
        try:
            parsed = safe_parse_json(call_deepseek(item["ticker"], item["text"]))
        except Exception as e:
            log.warning("LLM fail %s: %s", item["ticker"], e); parsed = None
        if parsed:
            parsed["ticker"] = item["ticker"]
            reg = _REGIME.get(str(parsed.get("market_regime", "neutral")).lower(), 0.0)
            # llm_score in [-1,1]: bull(+)/bear(-) * cường độ * độ tự tin -> feature, KHÔNG phải quyết định
            parsed["llm_score"] = reg * float(parsed.get("sentiment_intensity", 0) or 0) \
                                      * float(parsed.get("regime_confidence", 0) or 0)
            for k in ("key_concepts", "bull_points", "bear_points", "risk_flags"):
                if isinstance(parsed.get(k), list): parsed[k] = "; ".join(map(str, parsed[k]))
            rows.append(parsed)
    cols = ["ticker", "market_regime", "regime_confidence", "narrative", "key_concepts",
            "bull_points", "bear_points", "risk_flags", "sentiment_intensity", "summary", "llm_score"]
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)

llm_feat = run_llm_enrichment(llm_batch)

if len(llm_feat):
    print("=" * 72); print("MARKET CONTEXT (DeepSeek) — bull/bear + concept (feature, KHÔNG phải quyết định)")
    print("=" * 72)
    for _, r in llm_feat.iterrows():
        print(f"\n[{r['ticker']}] regime={r['market_regime']} (conf {r['regime_confidence']}) "
              f"| intensity {r['sentiment_intensity']} | llm_score {r['llm_score']:+.2f}")
        if r.get("narrative"):  print(f"   narrative: {r['narrative']}")
        if r.get("summary"):    print(f"   summary  : {r['summary']}")
        if r.get("risk_flags"): print(f"   ⚠ risks  : {r['risk_flags']}")
llm_feat.head() if len(llm_feat) else llm_feat


---
## Section 5 — Composite "meme heat score"

Normalize cross-sectional → weighted sum → rank. LLM catalyst hiển thị làm *"why"*.

In [ ]:
# Cell 5.1 — merge tất cả feature + normalize cross-sectional (z-score)
def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd and sd > 0 else s * 0.0

def assemble_features(meme_able, rvol_df, deltas, mention_feat, llm_feat):
    df = meme_able.set_index("ticker")
    for extra in (rvol_df, deltas, mention_feat, llm_feat):
        if extra is not None and len(extra):
            e = extra.set_index("ticker")
            e = e[[col for col in e.columns if col not in df.columns]]  # bỏ cột trùng (vd 'price')
            if len(e.columns):
                df = df.join(e, how="left")
    return df.reset_index()

# deltas có thể chưa tồn tại nếu chưa chạy IBKR; dùng placeholder rỗng
deltas = locals().get("deltas", pd.DataFrame(columns=["ticker", "borrow_fee_d", "shortable_d"]))
feat = assemble_features(meme_able, rvol_df, deltas, mention_feat, llm_feat)

# short đo THEO % thay vì absolute: shortable còn lại / float (thấp = tight = áp lực squeeze)
if "shortable_shares" in feat.columns and "float_shares" in feat.columns:
    feat["shortable_ratio"] = (pd.to_numeric(feat["shortable_shares"], errors="coerce")
                               / pd.to_numeric(feat["float_shares"], errors="coerce"))
    feat["z_shortable_inv"] = -zscore(feat["shortable_ratio"])   # ratio thấp -> z dương -> cộng heat

for col, src in [("z_short", "short_pct_float"), ("z_dtc", "days_to_cover"),
                 ("z_rvol", "rvol"), ("z_borrow_d", "borrow_fee_d")]:
    feat[col] = zscore(feat[src]) if src in feat else 0.0
feat.head()

In [ ]:
# Cell 5.2 — weighted sum -> heat score
w = CONFIG["weights"]

def safe_col(df, name, default=0.0):
    return pd.to_numeric(df[name], errors="coerce").fillna(default) if name in df else pd.Series(default, index=df.index)

def compute_heat(df):
    llm_term     = safe_col(df, "llm_score")   # [-1,1]: bull(+)/bear(-) * intensity * confidence

    heat = (
        w["short_pct"]     * safe_col(df, "z_short")
      + w["dtc"]           * safe_col(df, "z_dtc")
      + w["util"]          * (safe_col(df, "z_shortable_inv") if "z_shortable_inv" in df else safe_col(df, "z_short"))  # shortable utilization (% thay vì absolute)
      + w["rvol"]          * safe_col(df, "z_rvol")
      + w["mention_accel"] * safe_col(df, "mention_accel")
      + w["borrow_fee_d"]  * safe_col(df, "z_borrow_d")
      + w["llm_fomo"]      * llm_term
      + w["otm_call_vol"]  * safe_col(df, "otm_call_vol")
      - w["float_pen"]     * np.log1p(safe_col(df, "float_shares", 1e6)) / 20.0
      - w["mcap_pen"]      * np.log1p(safe_col(df, "market_cap", 1e9)) / 25.0
    )
    df = df.copy()
    df["heat"] = heat
    return df

feat = compute_heat(feat)
feat[["ticker", "heat"]].sort_values("heat", ascending=False)

In [ ]:
# Cell 5.2b — gán LEG (squeeze / momentum) bằng ngưỡng THÍCH NGHI (không fix cứng)
def assign_legs(df):
    a = CONFIG["adaptive"]
    wset = set(load_winners()["ticker"]) if len(load_winners()) else set()
    def wv(col): return winner_feature_values(col, wset, df)

    thr_short = adaptive_threshold(df.get("short_pct_float"), a["squeeze_pct"]["short_pct"], wv("short_pct_float"))
    thr_dtc   = adaptive_threshold(df.get("days_to_cover"),   a["squeeze_pct"]["dtc"],       wv("days_to_cover"))
    thr_accel = adaptive_threshold(df.get("mention_accel"),   a["momentum_pct"]["mention_accel"], wv("mention_accel"))
    thr_rvol  = adaptive_threshold(df.get("rvol"),            a["momentum_pct"]["rvol"],      wv("rvol"))

    s  = safe_col(df, "short_pct_float"); d = safe_col(df, "days_to_cover")
    ac = safe_col(df, "mention_accel");   rv = safe_col(df, "rvol")
    BIG = 1e18  # nếu ngưỡng NaN (thiếu data) -> coi như không ai qua leg đó
    squeeze  = (s  >= (thr_short if not np.isnan(thr_short) else BIG)) & \
               (d  >= (thr_dtc   if not np.isnan(thr_dtc)   else BIG))
    momentum = (ac >= (thr_accel if not np.isnan(thr_accel) else BIG)) & \
               (rv >= (thr_rvol  if not np.isnan(thr_rvol)  else BIG))

    out = df.copy()
    out["leg_squeeze"]  = squeeze
    out["leg_momentum"] = momentum
    out["leg"] = np.select([squeeze & momentum, squeeze, momentum],
                           ["both", "squeeze", "momentum"], default="none")
    log.info("Ngưỡng thích nghi | short%%≥%.3f dtc≥%.2f | accel≥%.2f rvol≥%.2f || squeeze=%d momentum=%d both=%d",
             thr_short, thr_dtc, thr_accel, thr_rvol,
             int((out.leg=="squeeze").sum()), int((out.leg=="momentum").sum()), int((out.leg=="both").sum()))
    return out

feat = assign_legs(feat)
feat[feat["leg"] != "none"][["ticker", "leg", "heat", "short_pct_float", "days_to_cover", "mention_accel", "rvol"]] \
    .sort_values("heat", ascending=False)


In [ ]:
# Cell 5.2c — SOI ngưỡng thích nghi vs phân phối universe (để tinh chỉnh CONFIG["adaptive"])
def inspect_thresholds(df):
    a = CONFIG["adaptive"]
    wset = set(load_winners()["ticker"]) if len(load_winners()) else set()
    specs = [
        ("short_pct_float", a["squeeze_pct"]["short_pct"],      "squeeze"),
        ("days_to_cover",   a["squeeze_pct"]["dtc"],            "squeeze"),
        ("mention_accel",   a["momentum_pct"]["mention_accel"], "momentum"),
        ("rvol",            a["momentum_pct"]["rvol"],          "momentum"),
    ]
    rows = []
    for col, q, leg in specs:
        s = pd.to_numeric(df.get(col, pd.Series(dtype=float)), errors="coerce").dropna()
        wv = winner_feature_values(col, wset, df)
        thr = adaptive_threshold(df.get(col), q, wv)
        n_pass = int((s >= thr).sum()) if not np.isnan(thr) else 0
        rows.append({
            "feature": col, "leg": leg,
            "n_data": len(s),
            "p10": round(_pct(s, .10), 3) if len(s) else np.nan,
            "p50": round(_pct(s, .50), 3) if len(s) else np.nan,
            "p90": round(_pct(s, .90), 3) if len(s) else np.nan,
            "target_pct": q,
            "threshold": round(thr, 3) if not np.isnan(thr) else np.nan,
            "winner_floor": round(_pct(wv, a["winner_floor_pct"]), 3) if len(wv) >= 3 else None,
            "n_winners_data": len(wv),
            "n_pass": n_pass,
        })
    out = pd.DataFrame(rows)
    log.info("Winners đã gắn: %d | tên: %s", len(wset), ", ".join(sorted(wset)[:12]) or "(chưa có)")
    log.info("Mẹo: threshold thấp hơn p50 = bar dễ; cao hơn p90 = bar gắt. Chỉnh target_pct trong CONFIG[\'adaptive\'].")
    return out

inspect_thresholds(feat)


In [ ]:
# Cell 5.3 — rank desc -> bảng watchlist nóng + leg + lý do (catalyst từ LLM = 'why')
def hot_table(df, n=20):
    cols = ["ticker", "leg", "heat", "market_regime", "short_pct_float", "days_to_cover", "rvol",
            "mention_accel", "borrow_fee_d", "narrative", "summary"]
    cols = [c for c in cols if c in df]
    sub = df[df["leg"] != "none"] if "leg" in df else df
    return sub.sort_values("heat", ascending=False)[cols].head(n).reset_index(drop=True)

hot = hot_table(feat)
hot


---
## Section 6 — Trigger / alert

**Rule-based, KHÔNG hỏi LLM.** Đây là input cho Bước 3 (entry logic).

In [ ]:
# Cell 6.1 — trigger = xác nhận intraday trên tên đã có leg (squeeze/momentum)
t = CONFIG["trigger"]

def evaluate_triggers(df):
    d = df.copy()
    rvol  = safe_col(d, "rvol"); price = safe_col(d, "price"); vwap = safe_col(d, "vwap")
    borrow_d = safe_col(d, "borrow_fee_d")
    leg = d["leg"] if "leg" in d else pd.Series("none", index=d.index)

    cond = (leg != "none") & (rvol > t["min_rvol"])
    if t["require_above_vwap"]:
        cond &= (price > vwap)
    if t.get("require_borrow_up"):
        cond &= (borrow_d > 0)
    d["TRIGGERED"] = cond
    return d[d["TRIGGERED"]].sort_values("heat", ascending=False)

triggered = evaluate_triggers(feat)
(triggered[["ticker", "leg", "heat", "rvol", "mention_accel", "borrow_fee_d"]]
 if len(triggered) else "Không có tên nào trigger lúc này.")


In [ ]:
# Cell 6.2 — emit alert (print/log/webhook) -> input cho entry logic
def emit_alerts(triggered):
    for _, row in triggered.iterrows():
        msg = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "ticker": row["ticker"],
            "heat": round(float(row["heat"]), 3),
            "rvol": round(float(safe_col(triggered, "rvol").loc[row.name]), 2),
            "catalyst": row.get("specific_catalyst", ""),
        }
        log.info("🔥 ALERT %s", json.dumps(msg, ensure_ascii=False))
        url = CONFIG["webhook_url"]
        if url:
            try:
                requests.post(url, json=msg, timeout=5)
            except Exception as e:
                log.warning("webhook fail: %s", e)

if len(triggered):
    emit_alerts(triggered)
else:
    log.info("Không có alert.")

---
## Section 7 — Persistence & loop

Snapshot point-in-time để backtest + có history tính Δ. Scheduler: S1 chạy 1×/sáng; S2–6 loop mỗi 2–5 phút trong giờ giao dịch.

In [ ]:
# Cell 7.1 — lưu snapshot heat-table theo timestamp để backtest point-in-time
HEAT_HISTORY = CACHE_DIR / "heat_history.parquet"

def persist_heat(df):
    snap = df.copy()
    snap["ts"] = datetime.now(timezone.utc)
    old = load_df(HEAT_HISTORY) if cache_exists(HEAT_HISTORY) else pd.DataFrame()
    snap = pd.concat([old, snap], ignore_index=True) if len(old) else snap
    save_df(snap, HEAT_HISTORY)
    log.info("Persisted heat snapshot (%d rows total)", len(snap))

persist_heat(feat)


In [ ]:
# Cell 7.2 — scheduler: S1 (structural) 1×/sáng; S2-6 (ignition) loop mỗi N phút
def run_ignition_cycle():
    """Một vòng fast-path KEYLESS: borrow Δ -> rvol -> social -> LLM top-N -> heat -> trigger."""
    rv   = fetch_rvol(watchlist)
    bor  = fetch_borrow(watchlist)
    if len(bor):
        dl = compute_deltas(append_snapshot(bor))
    else:
        dl = pd.DataFrame(columns=["ticker", "borrow_fee_d"])
    sdf  = fetch_social_mentions(set(watchlist))
    stw  = stocktwits_metrics(watchlist[:50])   # bổ sung, có thể rỗng
    mf   = update_mention_features(sdf, stw)
    batch = select_llm_candidates(mf, sdf, CONFIG["social"]["llm_top_n"])
    lf   = run_llm_enrichment(batch)
    f    = assemble_features(meme_able, rv, dl, mf, lf)
    for col, src in [("z_short", "short_pct_float"), ("z_dtc", "days_to_cover"),
                     ("z_rvol", "rvol"), ("z_borrow_d", "borrow_fee_d")]:
        f[col] = zscore(f[src]) if src in f else 0.0
    f    = compute_heat(f)
    f    = assign_legs(f)          # gán leg bằng ngưỡng thích nghi
    trig = evaluate_triggers(f)
    if len(trig):
        emit_alerts(trig)
    persist_heat(f)
    return f, trig

def main_loop(interval_min=3, max_cycles=None):
    """Gọi mỗi sáng: rebuild universe (S1) rồi loop ignition trong giờ giao dịch."""
    cycles = 0
    while max_cycles is None or cycles < max_cycles:
        try:
            run_ignition_cycle()
        except Exception as e:
            log.exception("cycle error: %s", e)
        cycles += 1
        time.sleep(interval_min * 60)

# Demo 1 vòng (bỏ comment để loop thật):
# main_loop(interval_min=3)
log.info("Scanner sẵn sàng (keyless). Chạy main_loop() trong giờ giao dịch.")

---
## 📘 Tham khảo tham số — lý do tồn tại & FIXED vs ADAPTIVE

Nguyên tắc: tham số **cấu trúc thị trường** (đổi theo regime) nên ADAPTIVE; tham số **rủi ro/cơ học** (bất biến) thì FIXED.

| Tham số | Cell | Giá trị | Lý do tồn tại / vì sao quan trọng | Fixed hay nên Adaptive? |
|---|---|---|---|---|
| `mcap_floor/ceiling` | 0.2 | 100M–30B | Định nghĩa "ai có thể thành meme" (band universe, KHÔNG phải trigger) | FIXED (band rộng, ít nhạy) |
| `squeeze_pct / momentum_pct` | 0.2 | .60/.80 | Ngưỡng vào leg = **percentile universe hôm nay** | ✅ ĐÃ ADAPTIVE (co theo regime) |
| `winner_floor_pct` | 0.2 | .10 | Nới ngưỡng đủ để từng bắt được winners (feedback loop) | ✅ ADAPTIVE theo winners |
| `rvol_th` | 8.2 | 3.0 | "Đã bắt đầu chạy chưa" — phân biệt ngày thường vs sự kiện | FIXED → **nên adaptive** (percentile vol) |
| `mom_th` | 8.2 | 0.05 | Lọc breakout giả đi ngang | FIXED (ổn) |
| `atr_mult` | 8.2 | 3.0 | Trailing rộng để không bị rũ bởi nhiễu meme | FIXED → **nên adaptive** theo vola regime |
| `max_hold` | 8.2 | 60 | Ôm trọn sóng meme (trước 10 = cắt non) | FIXED |
| `si_dtc_min` | 8.2 | 2.0 | Nhiên liệu squeeze (short kẹt) | FIXED → **nên adaptive** (mặt bằng short đổi) |
| `max_ext_ma` | 8.2 | 0.50 | Không đu đỉnh; data: ext cao = thua nhưng quá chặt cắt winner | FIXED (điểm ngọt; sweep 8.2b) |
| `hard_stop` | 8.2 | 0.08 | Giới hạn lỗ/lệnh — sống còn cho profile asymmetric | FIXED (rủi ro = nên cố định) |
| `hold_require_si` | 8.2 | True | Thesis phải CÒN đúng suốt hold, không chỉ entry | FIXED bool (logic) |
| `SI_PUB_LAG_D` | 9.1 | 12 | Tránh look-ahead: short interest công bố trễ ~12 ngày | FIXED (đặc tính dữ liệu) |

**Tóm tắt triết lý:** cái gì là *rủi ro/cơ học* (stop, sizing, lag dữ liệu) → FIXED. Cái gì là *ngưỡng so với đám đông* (short bao nhiêu là cao, volume bao nhiêu là spike) → ADAPTIVE, vì "cao/thấp" chỉ có nghĩa **tương đối với thời kỳ**. Hiện `rvol_th`, `atr_mult`, `si_dtc_min` vẫn FIXED — đó là nợ kỹ thuật, nên chuyển sang percentile/regime.

---
## Section 8 — Backtest (Daily, 2021→nay, Alpaca free)

**Đọc kỹ caveat — để không hiểu nhầm kết quả:**
- Dữ liệu: **daily bars Alpaca** (IEX, split-adjusted), từ 2021 (lúc hiện tượng meme thật sự bắt đầu).
- **Social & short-interest KHÔNG có lịch sử free** → backtest chỉ test phần *ignition tradable*: RVOL spike + giá trên trung bình 20 ngày + momentum 5 ngày. Đây là phần thực sự vào lệnh được; phần social/leg của live pipeline không thể tái dựng quá khứ miễn phí.
- **Survivorship bias:** vài tên đã huỷ niêm yết (BBBY, SPRT...) có thể thiếu data trên Alpaca → kết quả lệch *lạc quan*. Coi đây là minh hoạ pipeline, KHÔNG phải hiệu suất live thật.
- **Chưa tính phí/slippage.** Swing daily, giữ tối đa N ngày + trailing stop.

In [ ]:
# Cell 8.1 — fetch daily bars (Alpaca) cho rổ meme lịch sử 2021+, cache
import matplotlib.pyplot as plt

BT_CACHE = CACHE_DIR / "bt_daily.parquet"
# Rổ meme lịch sử 2021-2026 (gồm cả tên đã/đang chạy; tên delist có thể thiếu data).
BT_UNIVERSE = list(dict.fromkeys(SEED_SQUEEZE + [
    "EXPR","BBBY","NOK","SNDL","WISH","RIDE","IRNT","PROG","BBIG","HKD",
    "RDBX","APE","NVAX","LCID","RIVN","RBLX","COSM","MARK","GREE","SPRT",
    "ANY","CENN","MMAT","SST","TRKA","GFAI","ATNF","COMS","HUDI","CRKN",
]))

def fetch_daily(sym, start="2021-01-01"):
    url = f"https://data.alpaca.markets/v2/stocks/{sym}/bars"
    out, tok = [], None
    while True:
        params = {"timeframe": "1Day", "start": start, "limit": 10000,
                  "feed": "iex", "adjustment": "split"}
        if tok: params["page_token"] = tok
        r = requests.get(url, params=params, headers=ALPACA_DATA_HEADERS, timeout=20)
        if r.status_code != 200:
            return None
        js = r.json(); out += js.get("bars") or []
        tok = js.get("next_page_token")
        if not tok: break
    if not out:
        return None
    d = pd.DataFrame(out).rename(columns={"t": "date", "o": "open", "h": "high",
                                          "l": "low", "c": "close", "v": "vol"})
    d["date"] = pd.to_datetime(d["date"])
    return d[["date", "open", "high", "low", "close", "vol"]].sort_values("date").reset_index(drop=True)

def fetch_bt_data(symbols, start="2021-01-01", force=False):
    if cache_exists(BT_CACHE) and not force:
        log.info("Load bt data từ cache"); return load_df(BT_CACHE)
    frames = []
    for i, sym in enumerate(symbols):
        try:
            d = fetch_daily(sym, start)
            if d is not None and len(d) > 40:
                d["ticker"] = sym; frames.append(d)
        except Exception as e:
            log.debug("bt fetch %s: %s", sym, e)
        if i % 20 == 0:
            log.info("bt fetch %d/%d", i, len(symbols))
        time.sleep(0.15)
    big = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(big):
        save_df(big, BT_CACHE)
        log.info("bt cached: %d rows, %d tickers có data", len(big), big["ticker"].nunique())
    else:
        log.warning("bt data RỖNG (Alpaca data API?).")
    return big

bt = fetch_bt_data(BT_UNIVERSE)
print("tickers có data:", bt["ticker"].nunique() if len(bt) else 0,
      "/", len(BT_UNIVERSE))
bt.groupby("ticker").size().sort_values(ascending=False).head() if len(bt) else bt


In [ ]:
# Cell 8.2 — entry ignition + GATE (SI/follow-through/exhaustion) + EXIT (hard stop -8% + ATR trail)
BT = {
    # === ENTRY: "đã bắt đầu chạy chưa?" — tạo ra để định nghĩa thời điểm ignition ===
    "rvol_th":     3.0,    # FIXED. Volume >= 3x TB20. Vì sao quan trọng: đây là cái phân biệt "ngày thường" vs "có sự kiện".
                           #   Vì sao 3: <2 là nhiễu, 3 là mốc kinh điển. NÊN ADAPTIVE (percentile volume universe) — surface ở 10.2.
    "mom_th":      0.05,   # FIXED. Momentum 5 ngày > 5%. Quan trọng: lọc breakout giả đi ngang. Adaptive được theo vola.
    # === EXIT: để winner chạy, cắt loser nhanh (đây là nơi quyết định lời/lỗ thật sự) ===
    "atr_win":     14,     # FIXED. Cửa sổ ATR (chuẩn ngành). ATR = thước đo biến động -> stop co giãn theo vola từng mã.
    "atr_mult":    3.0,    # FIXED. Trailing = đỉnh - 3*ATR. Quan trọng: 3 đủ RỘNG để không bị rũ bởi nhiễu meme (vola cao).
                           #   NÊN ADAPTIVE theo regime: thị trường êm -> 2 (chặt), sôi động -> 4 (rộng).
    "ma_exit":     10,     # FIXED. Sau khi đã lời, gãy MA10 = hết đà -> thoát. Khoá lời nhưng vẫn cho chạy.
    "max_hold":    60,     # FIXED. Trần thời gian giữ. Trước là 10 (cắt non meme) -> 60 để ôm trọn sóng. Adaptive: theo độ dài sóng.
    # === RISK / SIZING: tách bạch để equity curve sạch ===
    "max_positions": 10,   # FIXED. Tối đa 10 vị thế đồng thời -> đa dạng hoá, không all-in 1 tên.
    "alloc":       10000.0,# FIXED. $ cố định/lệnh (KHÔNG compound) -> equity curve phản ánh edge, không nhiễu bởi tái phân bổ.
    "start_capital": 100000.0,  # vốn gốc để quy %.
}
# ==== FILTERS: tạo ra để GIẢM false-positive (no-meme = vào lệnh nhưng không có sóng thật) ====
# Lý do tồn tại: backtest cho thấy 46% lệnh là no-meme, ăn hết lời của nhóm reactive. Mỗi filter cắt 1 dạng rác.
# KHÔNG look-ahead: mọi điều kiện chỉ dùng bar <= i (ngày vào). A = bật hết.
FILTERS = {
    "use_si_gate":           True,   # FIXED bool. Chỉ vào khi CÓ nhiên liệu squeeze (short interest). Quan trọng nhất về mặt
                                     #   bản chất: phân biệt "spike ngẫu nhiên" vs "short bị kẹt". Cần chạy 9.1; trống -> tự TẮT.
    "si_dtc_min":            2.0,    # FIXED. Days-to-cover >= 2 = short cần >=2 ngày volume để thoát = kẹt. NÊN ADAPTIVE
                                     #   (percentile dtc của universe) vì mặt bằng short đổi theo thời kỳ.
    "require_followthrough": True,   # FIXED bool. Ngày vào phải đóng cửa XANH so hôm trước -> loại "pop 1 ngày rồi xịt".
    "max_ext_ma":            0.50,   # FIXED %. Bỏ nếu giá đã > +50% trên MA20 = đã đu đỉnh. Quan trọng: ext_ma cao = thua nhiều
                                     #   (xem 8.4 attribution). Nhưng quá chặt cắt winner bùng nổ -> .50 là điểm ngọt data; sweep 8.2b.
    "hard_stop":             0.08,   # FIXED %. Cắt -8% ngay = giới hạn lỗ/lệnh. Quan trọng cho profile "nhiều lỗ nhỏ, vài lãi to".
    # === GIÁM SÁT TRONG HOLD: điều kiện phải CÒN đúng mỗi ngày, không chỉ lúc entry (yêu cầu của bạn) ===
    "hold_monitor":          True,   # FIXED bool. Bật re-check thesis mỗi ngày giữ lệnh.
    "hold_require_si":       True,   # FIXED bool. Short-fuel mất trong hold -> bỏ luôn (không đợi giá stop).
    "hold_below_ma":         None,   # FIXED/None. set 20 -> gãy MA20 trong hold thì thoát; None = chỉ dựa SI + price stop.
}

def _atr(high, low, close, win):
    n = len(close); tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(high[i]-low[i], abs(high[i]-close[i-1]), abs(low[i]-close[i-1]))
    atr = np.full(n, np.nan)
    for i in range(win, n):
        atr[i] = tr[i-win+1:i+1].mean()
    return atr

def _si_gate_ok(sym, d, F):
    """point-in-time (si_asof có publication lag). Chưa chạy 9.1 -> degrade (True)."""
    if not F.get("use_si_gate"): return True
    simap = globals().get("si_map", None)
    if not simap: return True
    fn = globals().get("si_asof")
    si = fn(sym, d) if fn else None
    if not si: return False
    dtc, rising = si
    dtc_ok = (dtc is not None) and not (isinstance(dtc, float) and np.isnan(dtc)) and dtc >= F["si_dtc_min"]
    return bool(dtc_ok or rising)

def gen_trades(bt_df, filters=None):
    F = filters if filters is not None else FILTERS
    trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 40: continue
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        high = g["high"].values.astype(float); low = g["low"].values.astype(float)
        n = len(g); atr = _atr(high, low, close, BT["atr_win"])
        rvol = np.full(n, np.nan); rmean = np.full(n, np.nan); mom = np.full(n, np.nan)
        for i in range(20, n):
            avg = vol[i-20:i].mean()
            rvol[i] = vol[i]/avg if avg > 0 else np.nan
            rmean[i] = close[i-20:i].mean()
            mom[i] = close[i]/close[i-5] - 1
        i = 21   # cần i-1 cho follow-through
        while i < n - 1:
            ext = close[i]/rmean[i] - 1 if rmean[i] > 0 else 0
            ok = (rvol[i] > BT["rvol_th"]) and (close[i] > rmean[i]) and (mom[i] > BT["mom_th"])
            if F.get("require_followthrough"): ok = ok and (close[i] > close[i-1])
            if F.get("max_ext_ma") is not None: ok = ok and (ext <= F["max_ext_ma"])
            if ok and _si_gate_ok(sym, g["date"].iloc[i], F):
                entry_px = close[i]; peak = entry_px; exit_idx = min(i + BT["max_hold"], n - 1)
                for j in range(i + 1, exit_idx + 1):
                    if close[j] <= entry_px * (1 - F["hard_stop"]):   # HARD STOP trước tiên
                        exit_idx = j; break
                    if F.get("hold_monitor"):   # GIÁM SÁT ĐIỀU KIỆN MỖI NGÀY HOLD
                        if F.get("hold_require_si") and not _si_gate_ok(sym, g["date"].iloc[j], F):
                            exit_idx = j; break          # short-fuel biến mất -> bỏ
                        hb = F.get("hold_below_ma")
                        if hb:
                            ma_h = close[max(0, j - hb + 1):j + 1].mean()
                            if close[j] < ma_h:
                                exit_idx = j; break      # trend gãy trong hold -> bỏ
                    peak = max(peak, close[j])
                    atr_stop = peak - BT["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0)
                    ma = close[max(0, j - BT["ma_exit"] + 1):j + 1].mean()
                    if close[j] <= atr_stop: exit_idx = j; break
                    if close[j] > entry_px and close[j] < ma: exit_idx = j; break
                trades.append({"ticker": sym, "entry_date": g["date"].iloc[i], "exit_date": g["date"].iloc[exit_idx],
                               "entry_px": entry_px, "exit_px": close[exit_idx], "ret": close[exit_idx]/entry_px - 1,
                               "days": exit_idx - i, "rvol_in": rvol[i], "mom_in": mom[i], "ext_ma": ext})
                i = exit_idx + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

def run_portfolio(trades):
    rows, open_exits = [], []
    for _, tr in trades.iterrows():
        open_exits = [d for d in open_exits if d > tr["entry_date"]]
        if len(open_exits) >= BT["max_positions"]: continue
        open_exits.append(tr["exit_date"]); rows.append({**tr.to_dict(), "pnl": BT["alloc"] * tr["ret"]})
    taken = pd.DataFrame(rows)
    if len(taken):
        taken = taken.sort_values("exit_date").reset_index(drop=True)
        taken["equity"] = BT["start_capital"] + taken["pnl"].cumsum()
    return taken

if FILTERS["use_si_gate"] and not globals().get("si_map"):
    log.warning("SI gate BẬT nhưng chưa có si_map -> chạy Cell 9.1 TRƯỚC rồi chạy lại 8.2 để kích hoạt (tạm thời gate TẮT).")
trades = gen_trades(bt) if len(bt) else pd.DataFrame()
taken = run_portfolio(trades) if len(trades) else pd.DataFrame()
log.info("Signals: %d -> %d trades vào | hold tb %.1f ngày", len(trades), len(taken),
         taken["days"].mean() if len(taken) else 0)
taken.tail(10) if len(taken) else "Không có trade."


In [ ]:
# Cell 8.2b — MONITOR / A-B: bật-tắt từng filter, so performance (focus cohort REACTIVE)
# self-contained: định nghĩa detect_meme_events tại đây (8.5 chạy SAU nên không phụ thuộc)
if "detect_meme_events" not in globals():
    def detect_meme_events(bt_df, H=20, surge_th=0.50):
        evs = []
        for sym, g in bt_df.groupby("ticker"):
            g = g.sort_values("date").reset_index(drop=True)
            close = g["close"].values.astype(float); n = len(g); t = 0
            while t < n - 1:
                w = close[t+1:min(t+1+H, n)]
                if len(w) == 0: break
                if w.max()/close[t]-1 >= surge_th:
                    pk = t+1+int(np.argmax(w))
                    evs.append({"ticker": sym, "ignition_date": g["date"].iloc[t], "peak_date": g["date"].iloc[pk],
                                "base_px": close[t], "peak_px": close[pk], "run_pct": close[pk]/close[t]-1})
                    t = pk+1
                else: t += 1
        return pd.DataFrame(evs)
def _timing_label(row, events):
    ev = events[(events["ticker"] == row["ticker"]) &
                (events["ignition_date"] <= row["exit_date"]) &
                (events["peak_date"] >= row["entry_date"])]
    if len(ev) == 0: return "no-meme"
    e = ev.sort_values("ignition_date").iloc[0]
    return "early" if row["entry_date"] < e["ignition_date"] else ("reactive" if row["entry_date"] <= e["peak_date"] else "late")

def perf(F, events):
    td = gen_trades(bt, filters=F)
    if len(td) == 0: return {"n": 0, "win": np.nan, "pnl": 0, "nomeme%": np.nan,
                             "react_n": 0, "react_win": np.nan, "react_pnl": 0, "final": BT["start_capital"]}
    tk = run_portfolio(td).copy()
    tk["entry_date"] = pd.to_datetime(tk["entry_date"]); tk["exit_date"] = pd.to_datetime(tk["exit_date"])
    tk["timing"] = tk.apply(lambda r: _timing_label(r, events), axis=1) if len(events) else "?"
    r = tk["ret"]; react = tk[tk["timing"] == "reactive"]
    return {"n": len(tk), "win": round((r > 0).mean(), 3), "pnl": round(tk["pnl"].sum(), 0),
            "nomeme%": round((tk["timing"] == "no-meme").mean(), 3) if len(events) else np.nan,
            "react_n": len(react), "react_win": round((react["ret"] > 0).mean(), 3) if len(react) else np.nan,
            "react_pnl": round(react["pnl"].sum(), 0),
            "final": round(tk["equity"].iloc[-1], 0)}

if len(bt) == 0:
    print("Cần bt (8.1).")
else:
    events = detect_meme_events(bt) if "detect_meme_events" in globals() else pd.DataFrame()
    OFF = {"use_si_gate": False, "si_dtc_min": 2.0, "require_followthrough": False, "max_ext_ma": None,
           "hard_stop": 1.0, "hold_monitor": False, "hold_require_si": False, "hold_below_ma": None}
    # >>> CHỈNH Ở ĐÂY để tự A/B: bật/tắt từng dòng <<<
    configs = {
        "baseline (OFF hết)":  dict(OFF),
        "+hard_stop -8%":      {**OFF, "hard_stop": 0.08},
        "+follow-through":     {**OFF, "hard_stop": 0.08, "require_followthrough": True},
        "cap ext None":        {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": None},
        "cap ext .50":         {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.50},
        "cap ext .30":         {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.30},
        "cap ext .15":         {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.15},
        "A (.50 +SI gate)":    {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.50, "use_si_gate": True},
        "A + daily-monitor":   {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.50, "use_si_gate": True,
                                "hold_monitor": True, "hold_require_si": True},
        "A + monitor +MA20":   {**OFF, "hard_stop": 0.08, "require_followthrough": True, "max_ext_ma": 0.50, "use_si_gate": True,
                                "hold_monitor": True, "hold_require_si": True, "hold_below_ma": 20},
    }
    rows = [{"config": name, **perf(F, events)} for name, F in configs.items()]
    print(pd.DataFrame(rows).to_string(index=False))
    print("\\nĐọc: nomeme% càng THẤP càng tốt | react_* = cohort REACTIVE (cỗ máy lời) | final = vốn cuối")
    print("SI gate chỉ hiệu lực nếu đã chạy 9.1 (có si_map). Sửa dict 'configs' để thử tổ hợp khác.")


In [ ]:
# Cell 8.3 — kết quả: equity curve + stocks bắt được theo năm + final portfolio
if len(taken) == 0:
    print("Chưa có trade để vẽ. Kiểm tra Cell 8.1 (data) / nới BT thresholds.")
else:
    final = taken["equity"].iloc[-1]
    taken["year"] = pd.to_datetime(taken["entry_date"]).dt.year

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].plot(taken["exit_date"], taken["equity"], lw=1.6)
    ax[0].axhline(BT["start_capital"], ls="--", c="gray")
    ax[0].set_title(f"Equity curve (realized P&L) — final ${final:,.0f}")
    ax[0].set_ylabel("Portfolio ($)"); ax[0].grid(alpha=.3)

    yr = taken.groupby("year")["pnl"].sum()
    ax[1].bar(yr.index.astype(str), yr.values,
              color=["#2e9e5b" if v > 0 else "#c0392b" for v in yr.values])
    ax[1].set_title("P&L theo năm"); ax[1].axhline(0, c="k", lw=.8); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

    print("=== Stocks bắt được theo năm ===")
    for y, gg in taken.groupby("year"):
        names = list(dict.fromkeys(gg.sort_values("pnl", ascending=False)["ticker"].tolist()))
        print(f"{y}: {len(gg):2d} trades | {', '.join(names)}")

    wins = (taken["ret"] > 0).mean()
    best = taken.loc[taken["ret"].idxmax()]; worst = taken.loc[taken["ret"].idxmin()]
    print("\n=== Tổng kết ===")
    print(f"Số trade vào   : {len(taken)}")
    print(f"Win rate       : {wins:.0%}")
    print(f"Total return   : {(final/BT['start_capital']-1):+.1%}")
    print(f"Final portfolio: ${final:,.0f}  (vốn đầu ${BT['start_capital']:,.0f})")
    print(f"Best  : {best['ticker']} {best['ret']:+.0%} ({best['entry_date'].date()})")
    print(f"Worst : {worst['ticker']} {worst['ret']:+.0%} ({worst['entry_date'].date()})")
    print("\n[Caveat] social/short không có trong backtest; survivorship + chưa tính phí/slippage.")


In [ ]:
# Cell 8.4 — FEEDBACK LOG: metrics + indicator nào dẫn tới win/lose
if len(taken) == 0:
    print("Chưa có trade. Chạy 8.1->8.3 trước.")
else:
    r = taken["ret"].astype(float)
    wins, losses = taken[r > 0], taken[r <= 0]
    span_years = max((taken["entry_date"].max() - taken["entry_date"].min()).days / 365.25, 0.5)
    tpy = len(taken) / span_years

    win_rate   = len(wins) / len(taken)
    avg_win    = wins["ret"].mean() if len(wins) else 0.0
    avg_loss   = losses["ret"].mean() if len(losses) else 0.0
    payoff     = (avg_win / abs(avg_loss)) if avg_loss != 0 else float("inf")
    expectancy = r.mean()
    pf         = (wins["pnl"].sum() / abs(losses["pnl"].sum())) if (len(losses) and losses["pnl"].sum() != 0) else float("inf")
    sharpe_tr  = (r.mean() / r.std(ddof=1)) if len(r) > 1 and r.std(ddof=1) > 0 else float("nan")
    sharpe_ann = sharpe_tr * np.sqrt(tpy) if not np.isnan(sharpe_tr) else float("nan")

    eq = taken["equity"].values
    peak = np.maximum.accumulate(eq); maxdd = ((eq - peak) / peak).min()

    # max chuỗi thua liên tiếp
    streak = mx = 0
    for x in (r.values <= 0):
        streak = streak + 1 if x else 0; mx = max(mx, streak)

    print("=" * 60)
    print("PERFORMANCE")
    print("=" * 60)
    print(f"  Trades            : {len(taken)}  (~{tpy:.1f}/năm)")
    print(f"  Win rate          : {win_rate:.1%}   ({len(wins)}W / {len(losses)}L)")
    print(f"  Avg win / loss    : {avg_win:+.1%} / {avg_loss:+.1%}   payoff={payoff:.2f}")
    print(f"  Expectancy/trade  : {expectancy:+.2%}")
    print(f"  Profit factor     : {pf:.2f}")
    print(f"  Sharpe (per-trade): {sharpe_tr:.2f}    Sharpe (annualized~): {sharpe_ann:.2f}")
    print(f"  Max drawdown      : {maxdd:.1%}")
    print(f"  Max losing streak : {mx}")

    print("\n" + "=" * 60)
    print("ATTRIBUTION — indicator tại lúc vào lệnh: winners vs losers")
    print("=" * 60)
    inds = [("rvol_in", "RVOL"), ("mom_in", "Momentum 5d"), ("ext_ma", "Vượt MA20"), ("days", "Hold days")]
    rows = []
    for col, name in inds:
        if col not in taken: continue
        wm = wins[col].mean() if len(wins) else np.nan
        lm = losses[col].mean() if len(losses) else np.nan
        corr = taken[[col, "ret"]].corr().iloc[0, 1]
        # verdict: tách tốt nếu winners khác losers rõ + corr cùng dấu kỳ vọng
        sep = abs(wm - lm) / (abs(lm) + 1e-9)
        verdict = "PHÂN TÁCH TỐT" if (abs(corr) >= 0.12 and sep >= 0.15) else ("yếu/nhiễu" if abs(corr) < 0.05 else "vừa")
        rows.append({"indicator": name, "win_mean": round(wm, 3), "lose_mean": round(lm, 3),
                     "corr_ret": round(corr, 3), "verdict": verdict})
    attr = pd.DataFrame(rows)
    print(attr.to_string(index=False))

    print("\n--- Bucket (tercile) theo từng indicator: win-rate & avg-ret ---")
    def bucket_stats(col):
        d = taken.dropna(subset=[col]).copy()
        if d[col].nunique() < 3: return None
        try:
            d["b"] = pd.qcut(d[col], 3, labels=["low", "mid", "high"], duplicates="drop")
        except Exception:
            return None
        return d.groupby("b").agg(n=("ret", "size"),
                                  win_rate=("ret", lambda x: (x > 0).mean()),
                                  avg_ret=("ret", "mean")).round(3)
    for col, name in inds:
        g = bucket_stats(col)
        if g is not None:
            print(f"\n[{name}]"); print(g.to_string())

    # verdict tổng: indicator tách win/lose mạnh nhất theo spread win-rate high vs low
    print("\n" + "=" * 60)
    print("VERDICT — chỉnh param theo đây")
    print("=" * 60)
    best = []
    for col, name in inds:
        g = bucket_stats(col)
        if g is not None and "high" in g.index and "low" in g.index:
            spread = g.loc["high", "win_rate"] - g.loc["low", "win_rate"]
            best.append((name, col, spread))
    best.sort(key=lambda x: abs(x[2]), reverse=True)
    for name, col, spread in best:
        direction = "TĂNG ngưỡng" if spread > 0 else "tín hiệu NGƯỢC — cân nhắc bỏ/đảo"
        print(f"  {name:12s}: win-rate(high)-(low) = {spread:+.0%}  -> {direction} '{col}'")
    if best:
        top = best[0]
        print(f"\n  => Indicator đáng tin nhất: {top[0]} (spread {top[2]:+.0%}). "
              f"Siết ngưỡng cái này trong BT sẽ cải thiện win rate.")
    print("\n[Lưu ý] mẫu nhỏ -> đọc xu hướng, đừng over-fit. Social/short chưa có trong backtest.")

    # plot phân phối indicator: winners vs losers
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for k, (col, name) in enumerate([("rvol_in", "RVOL"), ("mom_in", "Momentum 5d"), ("ext_ma", "Vượt MA20")]):
        if col in taken:
            ax[k].hist(wins[col].dropna(), bins=20, alpha=.6, label="win", color="#2e9e5b")
            ax[k].hist(losses[col].dropna(), bins=20, alpha=.6, label="lose", color="#c0392b")
            ax[k].set_title(name); ax[k].legend(); ax[k].grid(alpha=.3)
    plt.tight_layout(); plt.show()


In [ ]:
# Cell 8.5 — Scanner timing (ĐÚNG: chỉ ghép run mà trade THỰC SỰ cưỡi) + detection-lag + report
MEME = {"H": 20, "surge_th": 0.50}   # "thành meme" = chạy >=50% trong 20 ngày tới

def detect_meme_events(bt_df, H=MEME["H"], surge_th=MEME["surge_th"]):
    evs = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        close = g["close"].values.astype(float); n = len(g)
        t = 0
        while t < n - 1:
            window = close[t+1:min(t+1+H, n)]
            if len(window) == 0: break
            run = window.max() / close[t] - 1
            if run >= surge_th:
                pk = t + 1 + int(np.argmax(window))
                evs.append({"ticker": sym, "ignition_date": g["date"].iloc[t],
                            "peak_date": g["date"].iloc[pk], "base_px": close[t],
                            "peak_px": close[pk], "run_pct": close[pk] / close[t] - 1})
                t = pk + 1
            else:
                t += 1
    return pd.DataFrame(evs)

if len(taken) == 0 or len(bt) == 0:
    print("Cần chạy 8.1->8.3 (có taken + bt) trước.")
else:
    events = detect_meme_events(bt)
    log.info("Phát hiện %d đợt 'thành meme' (>=%.0f%%/%dd) trên %d ticker",
             len(events), MEME["surge_th"]*100, MEME["H"],
             events["ticker"].nunique() if len(events) else 0)

    # ---------- (A) DETECTION LAG cấp sự kiện: dùng MỌI tín hiệu (trades), độc lập kết quả ----------
    sig = trades.copy()
    sig["entry_date"] = pd.to_datetime(sig["entry_date"])
    det = []
    for _, ev in events.iterrows():
        ign, pk = ev["ignition_date"], ev["peak_date"]
        s = sig[(sig["ticker"] == ev["ticker"]) &
                (sig["entry_date"] >= ign - pd.Timedelta(days=10)) &
                (sig["entry_date"] <= pk)]
        if len(s):
            first = s["entry_date"].min()
            det.append({"ticker": ev["ticker"], "ignition": ign.date(), "peak": pk.date(),
                        "run%": round(ev["run_pct"], 2),
                        "first_signal": first.date(),
                        "detect_lag_d": (first - ign).days})   # >0 = scanner TRỄ sau ignition
        else:
            det.append({"ticker": ev["ticker"], "ignition": ign.date(), "peak": pk.date(),
                        "run%": round(ev["run_pct"], 2), "first_signal": None, "detect_lag_d": np.nan})
    detdf = pd.DataFrame(det)

    # ---------- (B) TIMING cấp trade: CHỈ ghép run mà [ignition,peak] GIAO [entry,exit] ----------
    def matched_run(sym, e_date, x_date):
        e = events[(events["ticker"] == sym) &
                   (events["ignition_date"] <= x_date) &   # run bắt đầu trước khi trade thoát
                   (events["peak_date"] >= e_date)]        # và đỉnh sau khi trade vào -> CÓ overlap
        return e.sort_values("ignition_date").iloc[0] if len(e) else None

    recs = []
    for _, tr in taken.iterrows():
        e_date, x_date = pd.to_datetime(tr["entry_date"]), pd.to_datetime(tr["exit_date"])
        ev = matched_run(tr["ticker"], e_date, x_date)
        if ev is not None:
            lead_ign = (ev["ignition_date"] - e_date).days   # >0: vào TRƯỚC ignition
            timing = "early" if e_date < ev["ignition_date"] else ("reactive" if e_date <= ev["peak_date"] else "late")
            run_pct, peak_d = ev["run_pct"], ev["peak_date"]
        else:
            lead_ign = np.nan; timing = "no-meme"; run_pct = np.nan; peak_d = pd.NaT
        recs.append({
            "ticker": tr["ticker"], "entry_date": e_date.date(), "entry_px": round(tr["entry_px"], 2),
            "exit_date": x_date.date(), "exit_px": round(tr["exit_px"], 2), "hold_d": int(tr["days"]),
            "trigger": f"RVOL={tr['rvol_in']:.1f} mom={tr['mom_in']:+.0%} +MA20={tr['ext_ma']:+.0%}",
            "ret": round(tr["ret"], 3), "pnl": round(tr["pnl"], 0),
            "timing": timing, "lead_to_ign_d": lead_ign,
            "meme_run%": round(run_pct, 2) if pd.notna(run_pct) else np.nan,
            "meme_peak": peak_d.date() if pd.notna(peak_d) else None,
        })
    report = pd.DataFrame(recs)
    report.to_csv(CACHE_DIR / "trade_report.csv", index=False)

    print("=" * 100); print("REPORT TỪNG TRADE (lead_to_ign_d > 0 = vào TRƯỚC ignition; chỉ ghép run trade THỰC SỰ cưỡi)")
    print("=" * 100)
    with pd.option_context("display.max_rows", 300, "display.width", 220):
        print(report.sort_values("entry_date").to_string(index=False))

    print("\n" + "=" * 64); print("(A) SCANNER CÓ KỊP KHÔNG — detection lag cấp sự kiện meme"); print("=" * 64)
    detected = detdf.dropna(subset=["detect_lag_d"])
    print(f"  Đợt meme phát hiện được : {len(detected)}/{len(detdf)} ({len(detected)/max(len(detdf),1):.0%})  "
          f"(còn lại scanner BỎ LỠ hoàn toàn)")
    if len(detected):
        ml = detected["detect_lag_d"].median()
        print(f"  Detection lag (median)  : {ml:+.0f} ngày so với ignition  "
              f"-> {'TRỄ (trend-follower)' if ml > 0 else 'kịp/sớm'}")
        print(f"  % phát hiện TRƯỚC ignition: {(detected['detect_lag_d'] < 0).mean():.0%}")
        print("\n  10 đợt meme + lúc scanner bắt được:")
        print(detected.sort_values("ignition").head(10).to_string(index=False))

    print("\n" + "=" * 64); print("(B) TIMING cấp trade vs RETURN (early phải có return cao nếu thật sự sớm)"); print("=" * 64)
    tg = report.groupby("timing").agg(n=("ret", "size"),
                                      win_rate=("ret", lambda x: (x > 0).mean()),
                                      avg_ret=("ret", "mean"),
                                      total_pnl=("pnl", "sum")).round(3)
    print(tg.to_string())
    nomeme = (report["timing"] == "no-meme").mean()
    print(f"\n  False-positive (no-meme): {nomeme:.0%}  (vào lệnh nhưng KHÔNG cưỡi được đợt chạy >= {MEME['surge_th']:.0%})")

    print("\n" + "=" * 64); print("VERDICT"); print("=" * 64)
    if len(detected):
        ml = detected["detect_lag_d"].median()
        if ml > 1:
            print(f"  -> Scanner TRỄ ~{ml:.0f} ngày sau khi nổ. Đúng như bạn nói: đây là TREND-FOLLOWER,")
            print(f"     entry momentum cần cú chạy đã bắt đầu. Muốn vào sớm (kiểu 0.955) cần tín hiệu DẪN")
            print(f"     trước giá (accumulation / borrow-fee / social) — không có trong backtest daily free.")
    if "early" in tg.index and tg.loc["early", "avg_ret"] <= 0:
        print(f"  -> 'early' avg_ret={tg.loc['early','avg_ret']:+.1%} (<=0): kể cả số ít vào trước ignition")
        print(f"     cũng không lời ổn định -> entry daily-price không có alpha dự báo, chỉ là whipsaw.")
    print("\n  [Đã sửa] nhãn early/mid/late giờ chỉ tính khi trade THỰC SỰ overlap đợt meme;")
    print("  các 'early' giả (ghép nhầm run xa) đã bị dồn về no-meme.")
    print("Đã lưu: meme_cache/trade_report.csv")


---
## Section 9 — Leading signals: A (breakout sớm) + B (short interest) + C-proxy (Google Trends)

Mục tiêu: vào TRƯỚC khi momentum bùng, để bớt trend-following.
- **A** — breakout sớm khỏi nền chặt (rvol chỉ cần >1.5), vào ở cú đẩy đầu thay vì đợi RVOL>3.
- **B** — **Nasdaq short-interest history** (keyless): chỉ vào khi days-to-cover cao HOẶC short interest đang tăng → có nhiên liệu squeeze. Đây là dữ liệu *leading thật*, free.
- **C-proxy** — **Google Trends** search interest acceleration (thay cho Reddit lịch sử không có free). Bịa-proof: số thật từ Google.

> ⚠️ Nasdaq API + pytrends hay bị chặn/429. Mỗi cell có cache + degrade: thiếu nguồn nào thì backtest tự chạy trên các leg còn lại (tối thiểu A thuần price luôn chạy).
> ⛔ KHÔNG lấy dữ liệu lịch sử từ DeepSeek/LLM — nó bịa số.

In [ ]:
# Cell 9.1 — (B) Nasdaq short-interest history (keyless) + as-of helper
SI_CACHE = CACHE_DIR / "short_interest.parquet"

def fetch_short_interest(sym):
    url = f"https://api.nasdaq.com/api/quote/{sym}/short-interest"
    h = dict(CONFIG["http_headers"]); h["Accept"] = "application/json, text/plain, */*"
    r = requests.get(url, params={"assetClass": "stocks"}, headers=h, timeout=12)
    if r.status_code != 200:
        return None
    rows = (((r.json() or {}).get("data") or {}).get("shortInterestTable") or {}).get("rows")
    if not rows:
        return None
    d = pd.DataFrame(rows)
    d["date"] = pd.to_datetime(d["settlementDate"], errors="coerce")
    d["interest"] = pd.to_numeric(d["interest"].astype(str).str.replace(",", ""), errors="coerce")
    d["dtc"] = pd.to_numeric(d.get("daysToCover", np.nan).astype(str).str.replace(",", ""), errors="coerce")
    d["ticker"] = sym
    return d.dropna(subset=["date"])[["ticker", "date", "interest", "dtc"]].sort_values("date")

def fetch_all_si(symbols, force=False):
    if cache_exists(SI_CACHE) and not force:
        log.info("Load short-interest từ cache"); return load_df(SI_CACHE)
    frames = []
    for i, sym in enumerate(symbols):
        try:
            d = fetch_short_interest(sym)
            if d is not None and len(d): frames.append(d)
        except Exception as e:
            log.debug("SI %s: %s", sym, e)
        time.sleep(0.4)
        if i % 15 == 0: log.info("SI %d/%d", i, len(symbols))
    big = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(big): save_df(big, SI_CACHE); log.info("SI cached: %d ticker", big["ticker"].nunique())
    else: log.warning("SI RỖNG — Nasdaq API có thể chặn IP. Backtest 9.3 sẽ bỏ leg B.")
    return big

# smoke 1 mã trước
_si_smoke = fetch_short_interest("GME")
print("SMOKE GME short-interest:", "OK" if _si_smoke is not None else "FAIL (Nasdaq chặn?)")
if _si_smoke is not None: print(_si_smoke.tail(3).to_string(index=False))

si_all = fetch_all_si(list(bt["ticker"].unique()) if len(bt) else BT_UNIVERSE)
si_map = {tk: g.sort_values("date") for tk, g in si_all.groupby("ticker")} if len(si_all) else {}

SI_PUB_LAG_D = 12   # FINRA công bố short interest ~10-12 ngày SAU settlement -> tránh look-ahead
def si_asof(sym, d):
    """Trả (dtc, rising) tại bản short-interest ĐÃ CÔNG BỐ tính tới ngày d (point-in-time)."""
    g = si_map.get(sym)
    if g is None: return None
    dd = pd.to_datetime(d)
    if getattr(dd, "tzinfo", None) is not None: dd = dd.tz_localize(None)   # bt là tz-aware, SI tz-naive
    avail = g[g["date"] + pd.Timedelta(days=SI_PUB_LAG_D) <= dd]   # chỉ dùng bản đã public
    if len(avail) == 0: return None
    last = avail.iloc[-1]
    rising = bool(len(avail) >= 2 and last["interest"] > avail.iloc[-2]["interest"])
    return (last["dtc"], rising)

print("SI tickers có data:", len(si_map))


In [ ]:
# Cell 9.2 — (C-proxy) Google Trends attention history + as-of accel  [EXPERIMENTAL]
TRENDS_CACHE = CACHE_DIR / "trends.parquet"
try:
    from pytrends.request import TrendReq
except ImportError:
    TrendReq = None

def fetch_all_trends(symbols, start="2021-01-01", force=False):
    if cache_exists(TRENDS_CACHE) and not force:
        log.info("Load trends từ cache"); return load_df(TRENDS_CACHE)
    if TrendReq is None:
        log.warning("pytrends chưa cài (chạy Cell 0.0). Bỏ leg C."); return pd.DataFrame()
    tf = f"{start} {datetime.now():%Y-%m-%d}"
    frames = []
    for i, sym in enumerate(symbols):
        try:
            py = TrendReq(hl="en-US", timeout=(10, 25))
            py.build_payload([sym], timeframe=tf)
            d = py.interest_over_time()
            if len(d) and sym in d:
                t = d[[sym]].reset_index().rename(columns={"date": "date", sym: "interest"})
                t["ticker"] = sym; frames.append(t[["ticker", "date", "interest"]])
            time.sleep(2.0)   # pytrends rate-limit gắt
        except Exception as e:
            log.debug("trends %s: %s", sym, e)
            time.sleep(5.0)
        if i % 10 == 0: log.info("trends %d/%d", i, len(symbols))
    big = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(big): save_df(big, TRENDS_CACHE); log.info("trends cached: %d ticker", big["ticker"].nunique())
    else: log.warning("trends RỖNG (pytrends 429/chưa cài). Backtest 9.3 sẽ bỏ leg C.")
    return big

# smoke 1 mã
if TrendReq is not None:
    try:
        _py = TrendReq(hl="en-US"); _py.build_payload(["GME"], timeframe="2021-01-01 2021-03-01")
        _sm = _py.interest_over_time()
        print("SMOKE GME trends:", "OK" if len(_sm) else "rỗng")
    except Exception as e:
        print("SMOKE trends FAIL:", e)

# chỉ pull trends cho rổ nhỏ (pytrends chậm) — dùng tên có meme event hoặc seed
_trend_syms = list(bt["ticker"].unique())[:40] if len(bt) else SEED_SQUEEZE[:40]
trends_all = fetch_all_trends(_trend_syms)
trends_map = {tk: g.sort_values("date") for tk, g in trends_all.groupby("ticker")} if len(trends_all) else {}

def trends_asof(sym, d):
    """accel = interest tuần gần nhất <= d  -  trung bình 4 tuần trước. >0 = đang tăng tốc."""
    g = trends_map.get(sym)
    if g is None: return None
    dd = pd.to_datetime(d)
    if getattr(dd, "tzinfo", None) is not None: dd = dd.tz_localize(None)
    g = g[g["date"] <= dd]
    if len(g) < 5: return None
    cur = g["interest"].iloc[-1]; base = g["interest"].iloc[-5:-1].mean()
    return cur - base

print("Trends tickers có data:", len(trends_map))


In [ ]:
# Cell 9.3 — Backtest LEADING (A breakout + B short-interest + C trends) + so detection-lag
LEAD = {
    "lookback":   20,     # nền/đỉnh tham chiếu
    "range_tight": 0.80,  # nền đủ chặt: (max-min)/min < 80%
    "rvol_min":   1.5,    # volume chớm tăng (thấp hơn 3 của momentum -> vào SỚM hơn)
    "use_si":     True, "si_dtc_min": 2.0,   # leg B: dtc cao HOẶC short tăng
    "use_trends": True, "trends_accel_min": 0.0,
}

# --- GUARD: nếu chưa chạy 9.1/9.2/8.5 thì degrade thay vì crash ---
si_map     = globals().get("si_map", {})
trends_map = globals().get("trends_map", {})
if "si_asof" not in globals():
    def si_asof(sym, d): return None
if "trends_asof" not in globals():
    def trends_asof(sym, d): return None

def gen_leading_trades(bt_df):
    trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 40: continue
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        high = g["high"].values.astype(float); low = g["low"].values.astype(float)
        n = len(g); atr = _atr(high, low, close, BT["atr_win"])
        lb = LEAD["lookback"]
        i = lb
        while i < n - 1:
            base_hi = high[i-lb:i].max(); base_lo = low[i-lb:i].min()
            tight = (base_hi - base_lo) / base_lo < LEAD["range_tight"] if base_lo > 0 else False
            rvol_i = vol[i] / vol[i-20:i].mean() if vol[i-20:i].mean() > 0 else 0
            breakout = (close[i] > base_hi) and tight and (rvol_i > LEAD["rvol_min"])  # A
            if breakout:
                d = g["date"].iloc[i]
                ok_b = ok_c = (not LEAD["use_si"]) and (not LEAD["use_trends"])
                if LEAD["use_si"]:
                    si = si_asof(sym, d)
                    ok_b = bool(si and ((si[0] is not None and si[0] >= LEAD["si_dtc_min"]) or si[1]))
                if LEAD["use_trends"]:
                    tr = trends_asof(sym, d)
                    ok_c = bool(tr is not None and tr > LEAD["trends_accel_min"])
                if ok_b or ok_c or (not LEAD["use_si"] and not LEAD["use_trends"]):
                    entry_px = close[i]; peak = entry_px
                    exit_idx = min(i + BT["max_hold"], n - 1)
                    for j in range(i + 1, exit_idx + 1):
                        peak = max(peak, close[j])
                        atr_stop = peak - BT["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0)
                        ma = close[max(0, j - BT["ma_exit"] + 1):j + 1].mean()
                        if close[j] <= atr_stop: exit_idx = j; break
                        if close[j] > entry_px and close[j] < ma: exit_idx = j; break
                    trades.append({"ticker": sym, "entry_date": d, "exit_date": g["date"].iloc[exit_idx],
                                   "entry_px": entry_px, "exit_px": close[exit_idx],
                                   "ret": close[exit_idx]/entry_px - 1, "days": exit_idx - i,
                                   "rvol_in": rvol_i, "leg_b": ok_b, "leg_c": ok_c})
                    i = exit_idx + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

def detection_lag(trades_df, events):
    if trades_df is None or len(trades_df) == 0 or "entry_date" not in getattr(trades_df, "columns", []) or len(events) == 0:
        return (0, len(events) if events is not None else 0, np.nan)
    sig = trades_df.copy(); sig["entry_date"] = pd.to_datetime(sig["entry_date"])
    lags = []
    for _, ev in events.iterrows():
        s = sig[(sig["ticker"] == ev["ticker"]) &
                (sig["entry_date"] >= ev["ignition_date"] - pd.Timedelta(days=15)) &
                (sig["entry_date"] <= ev["peak_date"])]
        if len(s):
            lags.append((s["entry_date"].min() - ev["ignition_date"]).days)
    return (len(lags), len(events), np.median(lags) if lags else np.nan)

if len(bt) == 0:
    print("Cần bt (Cell 8.1).")
else:
    if "detect_meme_events" not in globals():
        raise RuntimeError("Hãy chạy Cell 8.5 trước (định nghĩa detect_meme_events).")
    events = detect_meme_events(bt)
    lead_trades = gen_leading_trades(bt)
    lead_taken  = run_portfolio(lead_trades) if len(lead_trades) else pd.DataFrame()

    print("=" * 64); print("SO SÁNH: MOMENTUM (8.2) vs LEADING (A+B+C, 9.3)"); print("=" * 64)
    for name, td in [("Momentum (đuổi RVOL>3)", trades), ("Leading (breakout+SI+trends)", lead_trades)]:
        if len(td):
            nd, ne, ml = detection_lag(td, events)
            wr = (td["ret"] > 0).mean(); ar = td["ret"].mean()
            print(f"  {name:32s}: {len(td):3d} trades | detect {nd}/{ne} đợt | "
                  f"lag median {ml:+.0f}d | win {wr:.0%} | avg_ret {ar:+.1%}")
        else:
            print(f"  {name:32s}: 0 trades")
    print("\n  lag âm = vào TRƯỚC ignition (mục tiêu). B dùng:", "có" if si_map else "KHÔNG (Nasdaq trống)",
          "| C dùng:", "có" if trends_map else "KHÔNG (trends trống)")

    if len(lead_taken):
        final = lead_taken["equity"].iloc[-1]
        print(f"\n  Leading portfolio final: ${final:,.0f}  ({(final/BT['start_capital']-1):+.1%}) | {len(lead_taken)} trades")
        print(lead_taken[["ticker","entry_date","entry_px","exit_date","exit_px","ret","leg_b","leg_c"]]
              .assign(entry_date=lambda x: pd.to_datetime(x.entry_date).dt.date,
                      exit_date=lambda x: pd.to_datetime(x.exit_date).dt.date).tail(15).to_string(index=False))


In [ ]:
# Cell 9.4 — BASE-RECLAIM entry (bắt đáy nền TRƯỚC cú nổ, kiểu AMC@0.955) + so entry-premium
# Ý tưởng: giá đang ở vùng đáy 60 ngày + reclaim MA20 lần đầu + volume CHỚM tăng (chưa bùng).
# KHÔNG look-ahead: mọi điều kiện dùng bar <= i; vào tại close[i]; exit quét forward = trade diễn tiến.
RECLAIM = {
    "ma": 20, "low_win": 60,
    "near_low_pct": 0.35,     # giá nằm trong 35% DƯỚI của range 60 ngày (còn ở nền, chưa chạy)
    "rvol_lo": 1.2, "rvol_hi": 5.0,   # volume chớm tăng, chưa explode
    "hard_stop": 0.08, "max_hold": 90,
}

def gen_base_reclaim_trades(bt_df):
    F = RECLAIM; trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < F["low_win"] + 5: continue
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        high = g["high"].values.astype(float); low = g["low"].values.astype(float)
        n = len(g); atr = _atr(high, low, close, BT["atr_win"])
        start = max(F["low_win"], F["ma"]) + 1
        i = start
        while i < n - 1:
            ma_now  = close[i-F["ma"]:i].mean()
            ma_prev = close[i-1-F["ma"]:i-1].mean()
            lo = low[i-F["low_win"]:i].min(); hi = high[i-F["low_win"]:i].max()
            pos = (close[i]-lo)/(hi-lo) if hi > lo else 1.0          # 0=đáy, 1=đỉnh range
            rvol_i = vol[i]/vol[i-20:i].mean() if vol[i-20:i].mean() > 0 else 0
            reclaim = (close[i] > ma_now) and (close[i-1] <= ma_prev)   # vượt MA20 LẦN ĐẦU
            near_base = pos <= F["near_low_pct"]
            early_vol = F["rvol_lo"] <= rvol_i <= F["rvol_hi"]
            if reclaim and near_base and early_vol:
                entry_px = close[i]; peak = entry_px; exit_idx = min(i + F["max_hold"], n - 1)
                for j in range(i + 1, exit_idx + 1):
                    if close[j] <= entry_px * (1 - F["hard_stop"]): exit_idx = j; break
                    peak = max(peak, close[j])
                    atr_stop = peak - BT["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0)
                    ma = close[max(0, j - BT["ma_exit"] + 1):j + 1].mean()
                    if close[j] <= atr_stop: exit_idx = j; break
                    if close[j] > entry_px and close[j] < ma: exit_idx = j; break
                trades.append({"ticker": sym, "entry_date": g["date"].iloc[i], "exit_date": g["date"].iloc[exit_idx],
                               "entry_px": entry_px, "exit_px": close[exit_idx], "ret": close[exit_idx]/entry_px - 1,
                               "days": exit_idx - i, "rvol_in": rvol_i, "pos_in_range": pos})
                i = exit_idx + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

def entry_premium_over_base(td, events):
    """median (entry_px / base_px_của_đợt_meme - 1): vào cao hơn ĐÁY NỀN bao nhiêu %. Thấp = vào gần đáy."""
    if len(td) == 0 or len(events) == 0: return np.nan
    prem = []
    for _, tr in td.iterrows():
        e = events[(events["ticker"] == tr["ticker"]) &
                   (events["ignition_date"] <= pd.to_datetime(tr["exit_date"])) &
                   (events["peak_date"] >= pd.to_datetime(tr["entry_date"]))]
        if len(e):
            base = e.sort_values("ignition_date").iloc[0]["base_px"]
            if base > 0: prem.append(tr["entry_px"]/base - 1)
    return np.median(prem) if prem else np.nan

if len(bt) == 0:
    print("Cần bt (8.1).")
else:
    events = detect_meme_events(bt)
    mom_td  = gen_trades(bt)                 # momentum (FILTERS hiện tại)
    base_td = gen_base_reclaim_trades(bt)    # base-reclaim (vào sớm)

    print("=" * 78); print("MOMENTUM (đuổi) vs BASE-RECLAIM (bắt đáy nền) — scale ý tưởng AMC@0.955")
    print("=" * 78)
    for name, td in [("Momentum (RVOL>3, đã chạy)", mom_td), ("Base-reclaim (đáy nền, sớm)", base_td)]:
        if len(td) == 0:
            print(f"  {name:30s}: 0 trades"); continue
        tk = run_portfolio(td)
        nd, ne, ml = detection_lag(td, events)
        prem = entry_premium_over_base(td, events)
        wr = (td["ret"] > 0).mean(); ar = td["ret"].mean()
        final = tk["equity"].iloc[-1] if len(tk) else BT["start_capital"]
        print(f"  {name:30s}: {len(td):3d} trades | detect {nd}/{ne} | lag median {ml:+.0f}d | "
              f"win {wr:.0%} | avg_ret {ar:+.1%} | entry+nền {prem:+.0%} | final ${final:,.0f}")
    print("\n  'entry+nền' = vào CAO HƠN đáy nền bao nhiêu %. Momentum thường +rất cao (đu đỉnh);")
    print("  base-reclaim phải THẤP hơn nhiều = vào gần đáy = giống mua AMC@0.955.")
    print("  lag âm = vào TRƯỚC ignition. Đánh đổi: base-reclaim win% thấp hơn nhưng vài runner cực lớn.")

    if len(base_td):
        print("\n  10 lệnh base-reclaim mới nhất:")
        print(base_td.assign(entry_date=lambda x: pd.to_datetime(x.entry_date).dt.date)[
              ["ticker","entry_date","entry_px","ret","days","rvol_in","pos_in_range"]].tail(10).to_string(index=False))


---
## Section 10 — Điều tra OVERFIT + truy tìm yếu tố ẩn

Câu hỏi: pnl đến từ EDGE bền hay vài cú may 2021? Trade gần đây thua = regime đổi hay thiếu factor?
- **10.1** Concentration + per-year + out-of-sample: pnl có dồn vào vài trade/1 năm không? Bỏ top-winner đi còn lời không?
- **10.2** Robustness surface: sweep tham số — best là **cao nguyên** (bền) hay **đỉnh nhọn** (overfit)?
- **10.3** Hidden-factor search: nhồi thêm nhiều feature point-in-time, xem feature nào THẬT SỰ tách win/lose (cái mình chưa dùng).

In [ ]:
# Cell 10.1 — Overfit test: concentration P&L + per-year + out-of-sample
if len(taken) == 0:
    print("Cần taken (8.2/8.3).")
else:
    tk = taken.copy()
    tk["entry_date"] = pd.to_datetime(tk["entry_date"]); tk["year"] = tk["entry_date"].dt.year
    total = tk["pnl"].sum()
    srt = tk.sort_values("pnl", ascending=False)

    print("=" * 60); print("(1) CONCENTRATION — pnl dồn vào bao nhiêu trade?"); print("=" * 60)
    for k in [1, 3, 5, 10]:
        share = srt.head(k)["pnl"].sum() / total if total else np.nan
        print(f"  Top {k:2d} trade  đóng góp {share:6.0%} tổng pnl")
    no_top5 = total - srt.head(5)["pnl"].sum()
    print(f"  Bỏ 5 winner lớn nhất -> pnl còn: ${no_top5:,.0f}  ({'CÒN LỜI' if no_top5>0 else 'HẾT LỜI -> dựa vào outlier'})")
    print(f"  Median trade ret: {tk['ret'].median():+.2%}  | Mean: {tk['ret'].mean():+.2%}  "
          f"({'median<0 -> sống nhờ outlier' if tk['ret'].median()<0 else 'median>0 -> lành mạnh'})")

    print("\n" + "=" * 60); print("(2) PER-YEAR — lời đều hay chỉ 1 mùa?"); print("=" * 60)
    yr = tk.groupby("year").agg(n=("ret","size"), win=("ret", lambda x:(x>0).mean()),
                                pnl=("pnl","sum")).round(3)
    print(yr.to_string())
    best_share = yr["pnl"].max() / total if total else np.nan
    print(f"  Năm tốt nhất chiếm {best_share:.0%} tổng pnl  "
          f"({'TẬP TRUNG 1 năm -> regime artifact' if best_share>0.6 else 'phân tán -> ổn hơn'})")

    print("\n" + "=" * 60); print("(3) OUT-OF-SAMPLE — nửa đầu (tune) vs nửa sau (test)"); print("=" * 60)
    mid = tk["entry_date"].quantile(0.5)
    IS, OOS = tk[tk["entry_date"] <= mid], tk[tk["entry_date"] > mid]
    for name, d in [("In-sample  (nửa đầu)", IS), ("Out-sample (nửa sau)", OOS)]:
        if len(d):
            print(f"  {name}: {len(d):3d} trade | win {(d['ret']>0).mean():.0%} | "
                  f"avg_ret {d['ret'].mean():+.1%} | pnl ${d['pnl'].sum():,.0f}")
    print("\n  VERDICT:")
    if no_top5 <= 0: print("   ⚠ Lời chỉ nhờ vài outlier -> KHÔNG phải edge bền.")
    if best_share > 0.6: print("   ⚠ Pnl dồn 1 năm -> phụ thuộc regime (vd meme-mania 2021), khó lặp lại.")
    if len(OOS) and OOS["pnl"].sum() <= 0 < IS["pnl"].sum():
        print("   ⚠ Out-of-sample LỖ trong khi in-sample lời -> OVERFIT rõ. Khớp việc AMC/WEN gần đây thua.")


In [ ]:
# Cell 10.2 — Robustness surface: best là CAO NGUYÊN (bền) hay ĐỈNH NHỌN (overfit)?
if len(bt) == 0:
    print("Cần bt.")
else:
    _OFF = {"use_si_gate": False, "si_dtc_min": 2.0, "require_followthrough": True, "max_ext_ma": None, "hard_stop": 0.08}
    ext_grid  = [0.15, 0.20, 0.30, 0.40, 0.50, 0.70, None]
    rvol_grid = [2.0, 3.0, 4.0, 5.0]
    grid = pd.DataFrame(index=[str(e) for e in ext_grid], columns=[str(r) for r in rvol_grid], dtype=float)
    _old = BT["rvol_th"]
    for e in ext_grid:
        for rv in rvol_grid:
            BT["rvol_th"] = rv
            td = gen_trades(bt, filters={**_OFF, "max_ext_ma": e})
            grid.loc[str(e), str(rv)] = run_portfolio(td)["pnl"].sum() if len(td) else 0.0
    BT["rvol_th"] = _old

    print("Total P&L theo (ext_ma cap = hàng) × (rvol_th = cột):  [$]")
    print((grid/1000).round(0).astype(int).to_string(), "   (đơn vị nghìn $)")
    flat = grid.values.flatten(); flat = flat[~np.isnan(flat)]
    best, med = np.nanmax(grid.values), np.nanmedian(flat)
    print(f"\n  Best ${best:,.0f} | Median ô ${med:,.0f} | tỉ lệ best/median = {best/med:.1f}x")
    print("  -> best/median CAO (vd >3x) = đỉnh nhọn = OVERFIT. Gần 1 = cao nguyên = bền.")
    try:
        fig, ax = plt.subplots(figsize=(7, 4))
        im = ax.imshow(grid.values.astype(float), aspect="auto", cmap="RdYlGn")
        ax.set_xticks(range(len(rvol_grid))); ax.set_xticklabels(rvol_grid)
        ax.set_yticks(range(len(ext_grid))); ax.set_yticklabels([str(e) for e in ext_grid])
        ax.set_xlabel("rvol_th"); ax.set_ylabel("max_ext_ma"); ax.set_title("P&L surface (xanh=cao)")
        plt.colorbar(im); plt.tight_layout(); plt.show()
    except Exception as e:
        print("  (skip heatmap:", e, ")")


In [ ]:
# Cell 10.3 — HIDDEN-FACTOR search: feature nào THẬT SỰ tách win/lose (kể cả cái chưa dùng)
# Nhồi feature point-in-time tại entry (KHÔNG look-ahead: chỉ bar <= i) + SPY regime.
def _build_spy_regime():
    try:
        spy = fetch_daily("SPY", "2020-06-01")
        if spy is None: return {}
        spy = spy.sort_values("date").reset_index(drop=True)
        spy["ma50"] = spy["close"].rolling(50).mean()
        return {d: int(c > m) if not np.isnan(m) else np.nan
                for d, c, m in zip(spy["date"], spy["close"], spy["ma50"])}
    except Exception:
        return {}

def enrich_trades(trades_df, bt_df):
    spy_map = _build_spy_regime()
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt_df.groupby("ticker")}
    rows = []
    for _, tr in trades_df.iterrows():
        g = idx.get(tr["ticker"])
        if g is None: continue
        pos = g.index[g["date"] == tr["entry_date"]]
        if len(pos) == 0: continue
        i = pos[0]
        if i < 60: continue
        c = g["close"].values; h = g["high"].values; lo = g["low"].values; v = g["vol"].values
        atr = _atr(h, lo, c, 14)
        rows.append({
            "ret": tr["ret"], "win": int(tr["ret"] > 0),
            "price":        c[i],
            "gap":          c[i]/c[i-1]-1,
            "rvol":         v[i]/v[i-20:i].mean() if v[i-20:i].mean()>0 else 0,
            "ext_ma":       c[i]/c[i-20:i].mean()-1,
            "mom5":         c[i]/c[i-5]-1,
            "atr_pct":      atr[i]/c[i] if not np.isnan(atr[i]) and c[i]>0 else np.nan,
            "dist_60low":   c[i]/lo[i-60:i].min()-1,
            "spy_bull":     spy_map.get(tr["entry_date"], np.nan),
        })
    return pd.DataFrame(rows)

if len(trades) == 0:
    print("Cần trades (8.2).")
else:
    fx = enrich_trades(trades, bt)
    feats = ["price","gap","rvol","ext_ma","mom5","atr_pct","dist_60low","spy_bull"]
    print("=" * 60); print(f"HIDDEN-FACTOR — {len(fx)} trade, tương quan feature vs return"); print("=" * 60)
    corr = fx[feats + ["ret"]].corr()["ret"].drop("ret").sort_values(key=lambda x: x.abs(), ascending=False)
    print(corr.round(3).to_string())
    print("\n  Win-rate theo tercile từng feature (low/mid/high):")
    for f in feats:
        d = fx.dropna(subset=[f])
        if d[f].nunique() < 3: continue
        try: d = d.assign(b=pd.qcut(d[f], 3, labels=["low","mid","high"], duplicates="drop"))
        except Exception: continue
        wr = d.groupby("b")["win"].mean()
        if "high" in wr and "low" in wr:
            print(f"   {f:11s}: low {wr.get('low',np.nan):.0%}  mid {wr.get('mid',np.nan):.0%}  high {wr.get('high',np.nan):.0%}  "
                  f"(spread {wr.get('high',0)-wr.get('low',0):+.0%})")
    try:
        from sklearn.tree import DecisionTreeClassifier
        d = fx.dropna(subset=feats)
        if len(d) > 30:
            clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20).fit(d[feats], d["win"])
            imp = pd.Series(clf.feature_importances_, index=feats).sort_values(ascending=False)
            print("\n  DecisionTree feature importance (depth 3):"); print(imp.round(3).to_string())
            print("\n  -> feature importance cao + spread lớn = factor THẬT. Nếu mọi feature đều yếu")
            print("     -> edge KHÔNG nằm ở giá/volume daily; yếu tố ẩn là short/borrow/social real-time (live layer).")
    except ImportError:
        print("\n  (cài scikit-learn để có feature importance: %pip install scikit-learn)")


In [ ]:
# Cell 10.4 — Condition persistence: losers có MẤT thesis trong lúc hold không? (kiểm chứng)
def hold_persistence(trades_df, bt_df):
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt_df.groupby("ticker")}
    rows = []
    for _, tr in trades_df.iterrows():
        g = idx.get(tr["ticker"])
        if g is None: continue
        e = g.index[g["date"] == tr["entry_date"]]; x = g.index[g["date"] == tr["exit_date"]]
        if len(e) == 0 or len(x) == 0: continue
        i, j0 = e[0], x[0]
        if j0 <= i or i < 20: continue
        c = g["close"].values
        days = range(i + 1, j0 + 1)
        above = [c[k] > c[max(0, k-20):k].mean() for k in days]           # còn trên MA20?
        trend_frac = float(np.mean(above)) if above else np.nan
        si_frac = np.nan
        if globals().get("si_map"):
            sok = [_si_gate_ok(tr["ticker"], g["date"].iloc[k], FILTERS) for k in days]
            si_frac = float(np.mean(sok)) if sok else np.nan
        rows.append({"win": int(tr["ret"] > 0), "ret": tr["ret"],
                     "trend_frac": trend_frac, "si_frac": si_frac})
    return pd.DataFrame(rows)

if len(taken) == 0:
    print("Cần taken (8.2).")
else:
    hp = hold_persistence(taken, bt)
    if len(hp):
        print("=" * 60); print("CONDITION PERSISTENCE — % ngày-hold điều kiện CÒN đúng (win vs lose)"); print("=" * 60)
        g = hp.groupby("win").agg(n=("ret", "size"),
                                  trend_frac=("trend_frac", "mean"),
                                  si_frac=("si_frac", "mean")).round(3)
        g.index = g.index.map({0: "LOSE", 1: "WIN"})
        print(g.to_string())
        wn, ls = hp[hp.win == 1], hp[hp.win == 0]
        if len(wn) and len(ls):
            dt = wn["trend_frac"].mean() - ls["trend_frac"].mean()
            print(f"\n  Winners ở trên MA20 nhiều hơn losers: {dt:+.0%} số ngày hold")
            if not hp["si_frac"].isna().all():
                ds = wn["si_frac"].mean() - ls["si_frac"].mean()
                print(f"  Winners giữ short-fuel (SI) nhiều hơn: {ds:+.0%} số ngày hold")
            print("\n  -> Nếu losers có trend_frac/si_frac THẤP hơn rõ = thesis GÃY trong hold")
            print("     => daily-monitor exit (8.2 hold_monitor) SẼ cải thiện. So 'A' vs 'A + daily-monitor' ở 8.2b.")
    else:
        print("Không đủ dữ liệu khớp entry/exit để tính.")


---
## Section 11 — Event study: tín hiệu gì xuất hiện TRƯỚC khi nổ? (giảm lag 16 ngày)

Backtest cho lag ~16 ngày (vào sau ignition). Để vào SỚM 15 ngày cần biết: 15 ngày trước ignition, feature nào đã nhúc nhích?
Cách làm: gióng tất cả đợt meme về T=0 (ignition), lấy TRUNG BÌNH quỹ đạo feature quanh đó → thấy cái gì "nóng lên" sớm.

In [ ]:
# Cell 11.1 — EVENT STUDY: quỹ đạo feature TRUNG BÌNH quanh ignition (T=0)
def pre_ignition_study(bt_df, events, pre=30, post=10):
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt_df.groupby("ticker")}
    keys = ["rvol", "dist_60low", "atr_pct", "vs_ma20"]
    mats = {k: [] for k in keys}
    for _, ev in events.iterrows():
        g = idx.get(ev["ticker"])
        if g is None: continue
        pos = g.index[g["date"] == ev["ignition_date"]]
        if len(pos) == 0: continue
        t = pos[0]
        if t < pre + 60 or t + post >= len(g): continue
        c = g["close"].values; h = g["high"].values; lo = g["low"].values; v = g["vol"].values
        atr = _atr(h, lo, c, 14)
        row = {k: [] for k in keys}
        for k in range(t - pre, t + post + 1):
            avgv = v[k-20:k].mean()
            row["rvol"].append(v[k]/avgv if avgv > 0 else np.nan)
            row["dist_60low"].append(c[k]/lo[k-60:k].min() - 1)
            row["atr_pct"].append(atr[k]/c[k] if not np.isnan(atr[k]) and c[k] > 0 else np.nan)
            row["vs_ma20"].append(c[k]/c[k-20:k].mean() - 1)
        for kk in keys: mats[kk].append(row[kk])
    xs = list(range(-pre, post + 1))
    avg = {kk: np.nanmean(np.array(mats[kk], dtype=float), axis=0) for kk in keys}
    return xs, avg, len(mats["rvol"])

if len(bt) == 0 or "detect_meme_events" not in globals():
    print("Cần bt + detect_meme_events (chạy 8.2b/8.5 trước).")
else:
    events = detect_meme_events(bt)
    xs, avg, nev = pre_ignition_study(bt, events)
    if nev == 0:
        print("Không đủ đợt meme có đủ lịch sử trước đó.")
    else:
        tbl = pd.DataFrame(avg, index=xs); tbl.index.name = "ngày_so_ignition"
        show = [t for t in [-20, -15, -10, -7, -5, -3, -1, 0, 3, 5] if t in xs]
        print(f"Event study trên {nev} đợt meme — giá trị TRUNG BÌNH quanh ignition (T=0):")
        print(tbl.loc[show].round(3).to_string())

        # tín hiệu sớm: ngày đầu rvol trung bình vượt 1.5 TRƯỚC ignition
        first = None
        for i, t in enumerate(xs):
            if t < 0 and not np.isnan(avg["rvol"][i]) and avg["rvol"][i] > 1.5:
                first = t; break
        ma_cross = None
        for i, t in enumerate(xs):
            if t < 0 and not np.isnan(avg["vs_ma20"][i]) and avg["vs_ma20"][i] > 0:
                ma_cross = t; break

        print("\n=== TÍN HIỆU SỚM (để vào trước ~15 ngày) ===")
        print(f"  RVOL trung bình vượt 1.5x lần đầu ở: T{first}" if first else "  RVOL không nhô rõ trước ignition")
        print(f"  Giá vượt lên trên MA20 lần đầu ở:   T{ma_cross}" if ma_cross else "  Giá chưa vượt MA20 trước ignition")
        print(f"  dist_60low tại T-15: {tbl.loc[-15,'dist_60low']:+.0%} (gần 0 = vẫn ở đáy nền = lý tưởng để gom)")
        print("\n  => GỢI Ý SIGNAL SỚM: rvol chớm 1.5 (chưa 3) + giá vừa cắt lên MA20 + còn gần đáy 60d")
        print("     = chính là logic Cell 9.4 (base-reclaim). Số ở trên xác nhận nó vào sớm hơn ~|T| ngày.")

        try:
            fig, ax = plt.subplots(1, 2, figsize=(14, 4))
            ax[0].plot(xs, avg["rvol"]); ax[0].axvline(0, c="r", ls="--"); ax[0].axhline(1.5, c="gray", ls=":")
            ax[0].set_title("RVOL TB quanh ignition"); ax[0].set_xlabel("ngày so ignition"); ax[0].grid(alpha=.3)
            ax[1].plot(xs, avg["dist_60low"], label="dist 60d-low", c="g")
            ax[1].plot(xs, avg["vs_ma20"], label="vs MA20", c="b")
            ax[1].axvline(0, c="r", ls="--"); ax[1].axhline(0, c="k", lw=.6)
            ax[1].set_title("Vị trí giá quanh ignition"); ax[1].legend(); ax[1].grid(alpha=.3)
            plt.tight_layout(); plt.show()
        except Exception as e:
            print("(skip plot:", e, ")")


In [ ]:
# Cell 11.2 — NHÌN PATH NHƯ GRAPH: quỹ đạo từng đợt (spaghetti) + median (không chỉ trung bình)
def collect_paths(bt_df, events, key, pre=30, post=10, maxn=120):
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt_df.groupby("ticker")}
    mats = []
    for _, ev in events.iterrows():
        g = idx.get(ev["ticker"])
        if g is None: continue
        pos = g.index[g["date"] == ev["ignition_date"]]
        if len(pos) == 0: continue
        t = pos[0]
        if t < pre + 60 or t + post >= len(g): continue
        c = g["close"].values; lo = g["low"].values; v = g["vol"].values
        seq = []
        for k in range(t - pre, t + post + 1):
            if key == "rvol":
                avgv = v[k-20:k].mean(); seq.append(v[k]/avgv if avgv > 0 else np.nan)
            elif key == "vs_ma20":
                seq.append(c[k]/c[k-20:k].mean() - 1)
            elif key == "dist_60low":
                seq.append(c[k]/lo[k-60:k].min() - 1)
        mats.append(seq)
        if len(mats) >= maxn: break
    return np.array(mats, dtype=float), list(range(-pre, post + 1))

if len(bt) == 0 or "detect_meme_events" not in globals():
    print("Cần bt + detect_meme_events.")
else:
    events = detect_meme_events(bt)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for c_i, (key, ttl, yl) in enumerate([("rvol","RVOL (volume thrust)",2),
                                          ("vs_ma20","Giá vs MA20 (pullback/breakout)",None),
                                          ("dist_60low","Khoảng cách trên đáy 60d",None)]):
        M, xs = collect_paths(bt, events, key)
        if len(M) == 0: continue
        for row in M[:80]:                      # spaghetti từng đợt
            ax[c_i].plot(xs, row, color="gray", alpha=0.08, lw=0.8)
        ax[c_i].plot(xs, np.nanmedian(M, axis=0), color="crimson", lw=2.2, label="median")
        ax[c_i].axvline(0, color="blue", ls="--", lw=1, label="ignition")
        if yl: ax[c_i].axhline(yl, color="green", ls=":", lw=1)
        if key == "vs_ma20": ax[c_i].axhline(0, color="k", lw=.6)
        ax[c_i].set_title(ttl); ax[c_i].set_xlabel("ngày so ignition"); ax[c_i].legend(); ax[c_i].grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("Đọc path: cú THRUST volume sớm (T-10) -> PULLBACK (vs_ma20 dip) -> BREAKOUT (T-1 rvol nổ).")
    print("Mua ở PULLBACK sau thrust = vào trước breakout. Đó là logic Cell 11.3.")


In [ ]:
# Cell 11.3 — ENTRY PATH-DEPENDENT: thrust -> pullback -> curl (mua dip sau cú volume đầu)
PATH = {
    "thrust_rvol":     2.0,   # cú volume thrust đầu (T-10 trong event study ~2.2)
    "thrust_lookback": 15,    # tìm thrust trong 15 ngày gần
    "thrust_min_ago":  2,     # thrust phải cách >=2 ngày (đã có thời gian pullback)
    "pullback_min":    0.03,  # giá đã rớt >=3% từ đỉnh sau thrust (đang ở dip)
    "rvol_curl_max":   4.0,   # vào khi rvol CÒN < 4 (chưa tới ngày nổ T-1 rvol~8)
    "max_hold":        60,
}

def _exit_after_entry(sym, g, close, atr, i, F, max_hold):
    entry_px = close[i]; peak = entry_px; exit_idx = min(i + max_hold, len(close) - 1)
    for j in range(i + 1, exit_idx + 1):
        if close[j] <= entry_px * (1 - F["hard_stop"]): return j
        if F.get("hold_monitor"):
            if F.get("hold_require_si") and not _si_gate_ok(sym, g["date"].iloc[j], F): return j
            hb = F.get("hold_below_ma")
            if hb and close[j] < close[max(0, j-hb+1):j+1].mean(): return j
        peak = max(peak, close[j])
        if close[j] <= peak - BT["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0): return j
        if close[j] > entry_px and close[j] < close[max(0, j-BT["ma_exit"]+1):j+1].mean(): return j
    return exit_idx

def gen_path_trades(bt_df, P=PATH, F=FILTERS):
    trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 70: continue
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        high = g["high"].values.astype(float); low = g["low"].values.astype(float)
        n = len(g); atr = _atr(high, low, close, BT["atr_win"])
        rvol = np.full(n, np.nan)
        for k in range(20, n):
            a = vol[k-20:k].mean(); rvol[k] = vol[k]/a if a > 0 else np.nan
        i = 25
        while i < n - 1:
            w1 = i - P["thrust_min_ago"]; w0 = max(20, i - P["thrust_lookback"])
            thrust_k = next((k for k in range(w1, w0 - 1, -1) if rvol[k] > P["thrust_rvol"]), None)
            if thrust_k is not None:
                post_high = close[thrust_k:i].max()
                pulled = close[i-1] <= post_high * (1 - P["pullback_min"])   # đã pullback
                curl   = close[i] > close[i-1]                              # đang ngóc lên
                early  = rvol[i] < P["rvol_curl_max"]                       # chưa nổ
                if pulled and curl and early and _si_gate_ok(sym, g["date"].iloc[i], F):
                    j = _exit_after_entry(sym, g, close, atr, i, F, P["max_hold"])
                    trades.append({"ticker": sym, "entry_date": g["date"].iloc[i], "exit_date": g["date"].iloc[j],
                                   "entry_px": close[i], "exit_px": close[j], "ret": close[j]/close[i]-1,
                                   "days": j - i, "rvol_in": rvol[i]})
                    i = j + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

if len(bt) == 0:
    print("Cần bt.")
else:
    events = detect_meme_events(bt)
    sets = [("Momentum (đuổi RVOL>3)", gen_trades(bt)),
            ("Base-reclaim (đáy nền)",  gen_base_reclaim_trades(bt)),
            ("PATH thrust->pullback",   gen_path_trades(bt))]
    print("=" * 86)
    print("3 KIỂU ENTRY — lag âm = vào trước ignition | entry+nền thấp = vào gần đáy | final = vốn cuối")
    print("=" * 86)
    for name, td in sets:
        if len(td) == 0: print(f"  {name:26s}: 0 trades"); continue
        tk = run_portfolio(td)
        nd, ne, ml = detection_lag(td, events)
        prem = entry_premium_over_base(td, events)
        final = tk["equity"].iloc[-1] if len(tk) else BT["start_capital"]
        print(f"  {name:26s}: {len(td):3d} tr | detect {nd}/{ne} | lag {ml:+.0f}d | "
              f"win {(td['ret']>0).mean():.0%} | avg {td['ret'].mean():+.1%} | entry+nền {prem:+.0%} | ${final:,.0f}")
    print("\n  Tinh chỉnh dict PATH (thrust_rvol, pullback_min, rvol_curl_max) rồi chạy lại để A/B.")
